# 서울시 그늘 형평성 연구 — 수정 전처리 파이프라인
**2021–2025년 / EPSG:5186 / 자치구 추세 + 2025 세부 후보지 분리**

이 수정본은 다음 원칙을 적용합니다.
- **2021–2025 형평성·추세의 기본 공간단위는 자치구**입니다. 고령 생활인구가 자치구 단위이므로 이를 500m 격자에 임의 배분하지 않습니다.
- **500m 격자는 필수 분석단위가 아닙니다.** 공간 LST와 세부 공급자료가 준비된 뒤 2025 후보지 탐색용으로만 선택적으로 생성합니다.
- ASOS는 **공통 기준시간 선정용**이며 공간 LST를 대체하지 않습니다.
- 가로수는 연도별 공식 자치구 총량을 추세 분석에 사용하고, 2025 위치자료는 세부 후보지의 공간분포 확인에 사용합니다.
- 가로수 반경 버퍼는 실제 그늘면적이 아니라 **가정 영향권**입니다. 2/3/4m 민감도 결과를 별도 저장합니다.
- 건물 그림자는 검증된/추정된 높이만 사용하고, 높이 미확인은 `unknown`으로 남깁니다.
- 스마트쉼터·인공그늘막은 **설치연도/스냅샷 연도보다 이전으로 소급하지 않습니다.**
- 2026 인공그늘막은 2025 공급으로 사용하지 않고, 2025 후보지의 2026 재검토에만 사용합니다.

원본은 수정하지 않으며 각 단계의 결과를 `data/processed`에 저장합니다.


In [28]:
# 최초 1회만 주석을 해제하여 실행합니다. Mac에서는 Python 3.11/3.12 환경을 권장합니다.
%pip install -U pandas numpy geopandas shapely pyogrio pyarrow openpyxl pvlib matplotlib nbformat
# OSM 건물 추출은 별도 도구(osmium 또는 pyosmium)가 필요할 수 있습니다.

Note: you may need to restart the kernel to use updated packages.


In [29]:
import sys
import platform

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Mac:", platform.machine())
print("macOS:", platform.mac_ver())

Python: 3.11.17 (main, Oct  1 2026, 20:09:52) [Clang 20.1.8 ]
Python executable: /opt/anaconda3/envs/datascience/bin/python
Mac: arm64
macOS: ('27.0.1', ('', '', ''), 'arm64')


In [30]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -U pyrosm

In [31]:
from pyrosm import OSM
import pyrosm

print("pyrosm:", pyrosm.__version__)
print("설치 성공")

pyrosm: 0.29.0
설치 성공


## 0장. 로컬 경로 설정 · 파일 자동 검색 · 캐시 설정

In [32]:
from pathlib import Path
import os, re, zipfile, warnings, unicodedata, shutil, json, datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box
from IPython.display import display

# [중요] 원본 데이터가 실제로 저장된 폴더를 지정할 수 있습니다.
# 예: RAW_DIR_OVERRIDE = Path.home() / "Downloads" / "서울시_연구데이터"
# 데이터가 여러 폴더에 흩어져 있다면 RAW_SEARCH_DIRS에 폴더를 추가하세요.
RAW_DIR_OVERRIDE = Path.home() / "Downloads" / "prof_stud_data"
RAW_SEARCH_DIRS = []
PROJECT_ROOT = Path.home() / "Documents" / "vscode_HB" / "교학"  # 예: Path.home() / "Documents" / "seoul-shade-equity"

# 현재 노트북이 어느 위치에서 실행되는지 먼저 확인합니다.
cwd = Path.cwd().resolve()
if PROJECT_ROOT is None:
    candidates = [cwd, cwd.parent] + list(cwd.parents)
    PROJECT_ROOT = next((p for p in candidates if (p/'data'/'raw').is_dir()), cwd)
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
OUT = PROJECT_ROOT/'data'/'processed'
REPORT_DIR = PROJECT_ROOT/'outputs'/'reports'
STAGING_DIR = PROJECT_ROOT/'.cache'/'flat_inputs'
for d in [OUT, REPORT_DIR, STAGING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 자동 탐색 후보: 프로젝트 내부, 다운로드, 문서, 데스크톱, iCloud Drive.
# 홈 전체를 무제한 탐색하지 않고, 사용자 데이터가 있을 만한 폴더만 검색합니다.
_home = Path.home()
_default_roots = [PROJECT_ROOT/'data'/'raw', cwd/'data'/'raw', cwd.parent/'data'/'raw',
                  _home/'Downloads', _home/'Documents', _home/'Desktop',
                  _home/'Library'/'Mobile Documents'/'com~apple~CloudDocs']
if RAW_DIR_OVERRIDE is not None:
    _roots = [Path(RAW_DIR_OVERRIDE).expanduser()]
else:
    _roots = [Path(p).expanduser() for p in RAW_SEARCH_DIRS] + _default_roots
# 중복 경로와 포함 관계를 정리합니다. 파일 수가 많으면 검색 시간이 걸릴 수 있습니다.
SEARCH_ROOTS = []
for _p in _roots:
    _p = _p.resolve()
    if _p.is_dir() and _p not in SEARCH_ROOTS:
        SEARCH_ROOTS.append(_p)
RAW_DIR = SEARCH_ROOTS[0] if SEARCH_ROOTS else PROJECT_ROOT/'data'/'raw'
print('현재 실행 위치:', cwd)
print('결과 저장 위치:', OUT)
print('원본 검색 위치:')
for _p in SEARCH_ROOTS: print('  -', _p)
if not SEARCH_ROOTS:
    print('⚠️ 검색 가능한 폴더가 없습니다. RAW_DIR_OVERRIDE를 실제 데이터 폴더로 지정하세요.')

# 실행할 챕터를 직접 선택합니다. 결과가 존재해도 재실행하려면 FORCE_RUN에 추가하세요.
FORCE_RUN = {'bus'}  # 버스 전처리만 기존 결과 무시하고 다시 실행

RUN_STAGES = {'boundary':True,'asos':True,'elderly_audit':True,'elderly':True,
              'trees':True,'tree_allocation':True,'building_audit':True,
              'building_gis':False,'osm_audit':True,'building_extract':True,'shadows':False,
              'bus':True,'facilities':True,'integration_audit':True}

CRS='EPSG:5186'; YEARS=range(2021,2026)
REF_HOUR=13
REF_MONTH,REF_DAY=8,1
TREE_RADIUS_M=3.0
TREE_RADIUS_SENSITIVITY_M=(2.0,3.0,4.0)
CREATE_CANDIDATE_GRID=False
CANDIDATE_GRID_SIZE_M=500
RUN_LARGE_BUILDINGS=RUN_STAGES['building_gis']
RUN_SHADOW=RUN_STAGES['shadows']

def norm(s):
    # macOS 파일명은 NFD(자모 분리)로 저장될 수 있어 NFC로 통일해 비교합니다.
    return unicodedata.normalize('NFC', str(s))

def raw_files():
    # 원본 폴더를 재귀적으로 검색합니다. 원본은 변경하지 않습니다.
    found = {}
    for root in SEARCH_ROOTS:
        for p in root.rglob('*'):
            if not p.is_file() or p.name.startswith('.'):
                continue
            # 노트북, 생성 결과물, 가상환경을 원본으로 인식하지 않습니다.
            if any(x in p.parts for x in ('processed', '.cache', '.venv', 'site-packages', 'node_modules')):
                continue
            if p.suffix.lower() not in ('.csv', '.xlsx', '.xls', '.shp', '.shx', '.dbf', '.prj', '.cpg', '.fix', '.zip', '.pbf'):
                continue
            found[str(p.resolve())] = p
    return sorted(found.values(), key=lambda p: norm(p.name))

# 실제 제공된 원본 파일명을 기준으로 만든 목록입니다. 파일명을 바꾸지 않아도 됩니다.
# Finder에 표시되는 이름과 내부 유니코드 형태가 달라도 동일하게 인식합니다.
EXPECTED_INPUTS = {
    'boundary': '서울시 상권분석서비스(영역-자치구).shp',
    'tree_positions': '2026 서울시 가로수 위치정보.xlsx',
    'tree_totals': '가로수+현황+2021_2025.csv',
    'gis_shape': 'AL_D010_11_20261009.shp',
    'gis_dbf_zip': 'AL_D010_11_20261009.dbf.zip',
    'building_register': '서울시 건축물대장 표제부.csv',
    'smart_shelter': '스마트쉼터 현황(20250520).xlsx',
    'sunshade': '폭염저감시설 목록_그늘막(260910).xlsx',
    'bus_definition': '컬럼정의_서울시 버스정류소 위치정보.xlsx',
}
for _year in YEARS:
    EXPECTED_INPUTS[f'asos_{_year}'] = f'ASOS_지면온도_{_year}.csv'
    EXPECTED_INPUTS[f'elderly_{_year}'] = f'LOCAL_PEOPLE_GU_{_year}.csv'
    EXPECTED_INPUTS[f'osm_{_year}'] = f'south-korea-{str(_year)[2:]}0101.osm.pbf'
    EXPECTED_INPUTS[f'bus_{_year}'] = (
        f'서울시버스정류소위치정보({_year}.01~12).csv' if _year<=2023
        else f'서울시버스정류소위치정보({_year}1209).xlsx')

def refresh_staging():
    # 원본 파일을 옮기지 않고 임시 심볼릭 링크만 만듭니다.
    for old in STAGING_DIR.iterdir():
        if old.is_symlink(): old.unlink()
    seen = {}
    for p in raw_files():
        key = norm(p.name).casefold()
        if key in seen:
            # 동일한 이름이 여러 폴더에 있으면 첫 파일을 사용하고 경고합니다.
            print('⚠️ 동일 파일명 중복:', p.name, '\n  ', seen[key], '\n  ', p)
            continue
        seen[key] = p
        link = STAGING_DIR / p.name
        if not link.exists(): link.symlink_to(p.resolve())
    print(f'원본 {len(seen)}개 파일 탐색 완료 (원본 변경 없음)')
    if not seen:
        print('❗ 원본 0개: 파일명이 아니라 폴더 위치 문제입니다.')
        print('   Finder에서 데이터 폴더를 찾은 뒤 RAW_DIR_OVERRIDE = Path("/Users/.../실제폴더") 로 지정하세요.')


refresh_staging()
DATA_DIR=STAGING_DIR

def find_file(keyword,suffix=None):
    # 정확한 파일명 우선 검색, 이후 부분일치 검색. 여러 개면 잘못 선택하지 않고 중단합니다.
    q=norm(keyword).casefold()
    files=[p for p in DATA_DIR.iterdir() if p.is_file()]
    if suffix is not None:
        files=[p for p in files if norm(p.name).casefold().endswith(norm(suffix).casefold())]
    exact=[p for p in files if norm(p.name).casefold()==q]
    found=exact if exact else [p for p in files if q in norm(p.name).casefold()]
    if not found:
        raise FileNotFoundError(f'파일을 찾지 못했습니다: {keyword} / data/raw 하위 폴더를 확인하세요.')
    if len(found)>1:
        raise ValueError(f'검색 결과가 여러 개입니다. 정확한 파일명을 사용하세요: {keyword} -> {[norm(p.name) for p in found]}')
    return found[0]

def input_file(key):
    # 파일명이 비슷한 연도 자료를 잘못 읽지 않도록 명시된 이름으로 찾습니다.
    return find_file(EXPECTED_INPUTS[key])

def input_report():
    rows=[]
    for key, filename in EXPECTED_INPUTS.items():
        try:
            p=input_file(key)
            rows.append({'데이터':key,'실제 파일명':norm(p.name),'상태':'찾음','경로':str(p.resolve())})
        except (FileNotFoundError,ValueError) as e:
            rows.append({'데이터':key,'실제 파일명':filename,'상태':'확인 필요','경로':str(e)})
    return pd.DataFrame(rows)

manifest=input_report()
display(manifest[['데이터','실제 파일명','상태']])
manifest.to_csv(REPORT_DIR/'input_filename_check.csv',index=False,encoding='utf-8-sig')
print('원본 파일명 점검표:',REPORT_DIR/'input_filename_check.csv')
if not (manifest['상태']=='찾음').any():
    raise FileNotFoundError('원본 파일을 하나도 연결하지 못했습니다. Downloads/prof_stud_data 내부 파일 구조를 확인하세요.')

def read_csv_auto(path,**kwargs):
    for enc in ('utf-8-sig','cp949','euc-kr'):
        try:return pd.read_csv(path,encoding=enc,**kwargs)
        except UnicodeDecodeError:pass
    raise UnicodeError(f'인코딩 확인 필요: {path}')
def save_table(df,name):
    path=OUT/name
    if path.suffix=='.parquet':df.to_parquet(path,index=False)
    else:df.to_csv(path,index=False,encoding='utf-8-sig')
    print('저장:',path.name,'행:',len(df))
def save_geo(gdf,name):
    path=OUT/name
    if path.exists():path.unlink()
    gdf.to_file(path,driver='GPKG')
    print('저장:',path.name,'도형:',len(gdf))
def should_run(stage,outputs):
    if not RUN_STAGES.get(stage,True):
        print('사용자 설정으로 생략:',stage);return False
    if 'all' in FORCE_RUN or stage in FORCE_RUN:return True
    # 결과 파일이 모두 있으면 해당 단계를 생략합니다.
    return not (outputs and all((OUT/x).exists() and (OUT/x).stat().st_size > 0 for x in outputs))
def mark_done(stage):
    (REPORT_DIR/f'{stage}_last_run.json').write_text(json.dumps({
        'stage':stage,'completed_at':datetime.datetime.now().isoformat(),
        'note':'입력 파일이 바뀌었다면 FORCE_RUN으로 재실행하세요.'},ensure_ascii=False,indent=2),encoding='utf-8')
    print('챕터 실행 종료:',stage)

display(pd.DataFrame([{'파일':norm(p.name),'폴더':str(p.parent),
                      '크기_MB':round(p.stat().st_size/1e6,1)} for p in raw_files()]))
print('프로젝트:',PROJECT_ROOT,'\n원본:',RAW_DIR,'\n결과:',OUT)
# 뒤 장을 단독 실행할 때 경로를 임의로 만들지 않고 선행 설정을 확인합니다.
def require_setup(required, stage):
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(f'{stage}: 선행 설정 없음 {missing}. 커널 재시작 후에는 0장부터 실행하세요.')


현재 실행 위치: /Users/hongbini/Documents/vscode_HB/교학
결과 저장 위치: /Users/hongbini/Documents/vscode_HB/교학/data/processed
원본 검색 위치:
  - /Users/hongbini/Downloads/prof_stud_data
원본 43개 파일 탐색 완료 (원본 변경 없음)


,데이터,실제 파일명,상태
0,boundary,서울시 상권분석서비스(영역-자치구).shp,찾음
1,tree_positions,2026 서울시 가로수 위치정보.xlsx,찾음
2,tree_totals,가로수+현황+2021_2025.csv,찾음
3,gis_shape,AL_D010_11_20261009.shp,찾음
4,gis_dbf_zip,AL_D010_11_20261009.dbf.zip,찾음
5,building_register,서울시 건축물대장 표제부.csv,찾음
6,smart_shelter,스마트쉼터 현황(20250520).xlsx,찾음
7,sunshade,폭염저감시설 목록_그늘막(260910).xlsx,찾음
8,bus_definition,컬럼정의_서울시 버스정류소 위치정보.xlsx,찾음
9,asos_2021,ASOS_지면온도_2021.csv,찾음


원본 파일명 점검표: /Users/hongbini/Documents/vscode_HB/교학/outputs/reports/input_filename_check.csv


,파일,폴더,크기_MB
0,2026 서울시 가로수 위치정보.xlsx,/Users/hongbini/Downloads/prof_stud_data,12.9
1,AL_D010_11_20261009.dbf,/Users/hongbini/Downloads/prof_stud_data,1262.8
2,AL_D010_11_20261009.dbf.zip,/Users/hongbini/Downloads/prof_stud_data,57.2
3,AL_D010_11_20261009.fix,/Users/hongbini/Downloads/prof_stud_data,8.3
4,AL_D010_11_20261009.prj,/Users/hongbini/Downloads/prof_stud_data,0.0
5,AL_D010_11_20261009.shp,/Users/hongbini/Downloads/prof_stud_data,129.7
6,AL_D010_11_20261009.shx,/Users/hongbini/Downloads/prof_stud_data,5.6
7,ASOS_지면온도_2021.csv,/Users/hongbini/Downloads/prof_stud_data,0.3
8,ASOS_지면온도_2022.csv,/Users/hongbini/Downloads/prof_stud_data,0.3
9,ASOS_지면온도_2023.csv,/Users/hongbini/Downloads/prof_stud_data,0.3


프로젝트: /Users/hongbini/Documents/vscode_HB/교학 
원본: /Users/hongbini/Downloads/prof_stud_data 
결과: /Users/hongbini/Documents/vscode_HB/교학/data/processed


In [33]:
# 0장 추가 점검: 실제 원본 폴더와 매칭 결과를 확인합니다.
print('원본 경로:', RAW_DIR_OVERRIDE)
print('원본 경로 존재:', RAW_DIR_OVERRIDE.is_dir())
print('발견한 원본 파일 수:', len(raw_files()))
print('매칭 성공:', int((manifest['상태']=='찾음').sum()), '/', len(manifest))
display(manifest[manifest['상태']!='찾음'][['데이터','실제 파일명','경로']])
# 경계 파일은 SHP/SHX/DBF/PRJ 세트가 동일 폴더에 있어야 합니다.
boundary_path = next((p for p in raw_files() if norm(p.name)==EXPECTED_INPUTS['boundary']), None)
if boundary_path:
    for ext in ['.shp','.shx','.dbf','.prj']:
        print('경계 구성파일',ext, (boundary_path.with_suffix(ext)).exists())


원본 경로: /Users/hongbini/Downloads/prof_stud_data
원본 경로 존재: True
발견한 원본 파일 수: 43
매칭 성공: 29 / 29


,데이터,실제 파일명,경로


경계 구성파일 .shp True
경계 구성파일 .shx True
경계 구성파일 .dbf True
경계 구성파일 .prj True


## 1장. 서울시 자치구 경계
고령 생활인구가 자치구 단위이므로 **자치구를 2021–2025 주 분석단위**로 사용합니다. 500m 격자는 이 단계의 필수 산출물이 아닙니다.


In [34]:
# boundary: 2021~2025 주 분석단위인 서울 25개 자치구 경계를 생성합니다.
if should_run('boundary',['districts.gpkg']):
    boundary_path=input_file('boundary')
    districts=gpd.read_file(boundary_path).to_crs(CRS)
    print('원본 컬럼:',districts.columns.tolist(),'행:',len(districts))
    districts=districts.loc[districts.geometry.notna() & ~districts.geometry.is_empty].copy()
    districts['geometry']=districts.geometry.make_valid()
    assert len(districts)==25,'25개 자치구가 아닙니다. 원본 행·중복을 확인하세요.'
    code_col=next((c for c in districts if any(k in norm(c).lower() for k in ['signgu_cd','sig_cd','자치구코드','adm_cd','signgu_code','시군구코드'])),None)
    name_col=next((c for c in districts if any(k in norm(c).lower() for k in ['signgu_nm','자치구명','구명'])),None)
    if code_col is None:raise ValueError('자치구 코드 컬럼을 확인해 code_col을 지정하세요.')
    districts['district_code']=districts[code_col].astype(str).str.replace(r'\.0$','',regex=True).str.zfill(5)
    districts['district_name']=districts[name_col].astype(str).str.strip() if name_col else districts['district_code']
    districts['district_area_m2']=districts.area
    if districts['district_code'].duplicated().any():raise ValueError('자치구 코드 중복')
    save_geo(districts[['district_code','district_name','district_area_m2','geometry']],'districts.gpkg')
    mark_done('boundary')
else:print('기존 결과 재사용: boundary')

# 선택 단계: 공간 LST가 준비된 뒤 2025 후보지 탐색에만 사용합니다.
if CREATE_CANDIDATE_GRID:
    districts=gpd.read_file(OUT/'districts.gpkg').to_crs(CRS)
    size=CANDIDATE_GRID_SIZE_M
    xmin,ymin,xmax,ymax=districts.total_bounds
    xs=np.arange(np.floor(xmin/size)*size,xmax,size);ys=np.arange(np.floor(ymin/size)*size,ymax,size)
    candidate_grid=gpd.GeoDataFrame({'grid_id':[f'G{int(x)}_{int(y)}' for x in xs for y in ys]},geometry=[box(x,y,x+size,y+size) for x in xs for y in ys],crs=CRS)
    candidate_grid=gpd.overlay(candidate_grid,districts[['district_code','geometry']],how='intersection',keep_geom_type=True)
    candidate_grid['cell_area_m2']=candidate_grid.area
    save_geo(candidate_grid,f'candidate_grid_{size}m.gpkg')
    print('※ 이 격자에 고령인구를 임의 배분하지 않습니다.')


원본 컬럼: ['SIGNGU_CD', 'SIGNGU_NM', 'XCNTS_VALU', 'YDNTS_VALU', 'RELM_AR', 'geometry'] 행: 25
저장: districts.gpkg 도형: 25
챕터 실행 종료: boundary


In [35]:
# 뒤 챕터에서 사용하므로 캐시를 건너뛴 경우에도 자치구 경계를 메모리에 불러옵니다.
if (OUT/'districts.gpkg').exists():districts=gpd.read_file(OUT/'districts.gpkg').to_crs(CRS)


## 2장. ASOS 지면온도 — 2021~2025년 기준시간

In [36]:
# asos: 2021~2025 서울(108) 여름철 지면온도로 기준시간을 자동 선정합니다.
if should_run('asos',['asos_summer_clean.parquet','asos_hourly.csv','reference_time.csv']):
    asos_parts=[]
    for year in YEARS:
        p=input_file(f'asos_{year}')
        df=read_csv_auto(p)
        required=['지점','일시','지면온도(°C)']
        missing=[c for c in required if c not in df.columns]
        if missing: raise ValueError(f'{year} ASOS 필수 컬럼 없음:{missing}')
        df['datetime']=pd.to_datetime(df['일시'],errors='coerce')
        df['temperature_c']=pd.to_numeric(df['지면온도(°C)'],errors='coerce')
        station=pd.to_numeric(df['지점'],errors='coerce')
        df=df[(station==108)&df['datetime'].dt.month.between(6,8)].copy()
        df=df.dropna(subset=['datetime','temperature_c'])
        df['year']=year
        df['hour']=df['datetime'].dt.hour
        asos_parts.append(df[['year','datetime','hour','temperature_c']])
    asos=pd.concat(asos_parts,ignore_index=True)
    if asos.empty: raise ValueError('서울108·6~8월 ASOS 지면온도 자료가 없습니다.')
    save_table(asos,'asos_summer_clean.parquet')

    hourly=(asos.groupby(['year','hour'],as_index=False)
            .agg(n=('temperature_c','count'),mean=('temperature_c','mean'),median=('temperature_c','median'),
                 maximum=('temperature_c','max'),p95=('temperature_c',lambda s:s.quantile(.95))))
    save_table(hourly,'asos_hourly.csv')

    pooled=(asos.groupby('hour',as_index=False)
            .agg(n=('temperature_c','count'),mean=('temperature_c','mean'),median=('temperature_c','median'),
                 maximum=('temperature_c','max'),p95=('temperature_c',lambda s:s.quantile(.95))))
    year_coverage=hourly.groupby('hour')['year'].nunique().rename('year_coverage').reset_index()
    pooled=pooled.merge(year_coverage,on='hour',how='left')
    eligible=pooled[pooled['year_coverage']==len(list(YEARS))].copy()
    if eligible.empty: raise ValueError('5개년 모두 관측된 공통 시간대가 없습니다.')

    # 기준시간: 5개년 모두 자료가 있는 시간 중 여름철 평균 지면온도가 가장 높은 시간.
    # 평균이 같으면 p95가 높은 시간을 우선합니다.
    selected=eligible.sort_values(['mean','p95'],ascending=[False,False]).iloc[0]
    ref_hour=int(selected['hour'])
    yearly_peak=(hourly.loc[hourly.groupby('year')['mean'].idxmax(),['year','hour','mean']]
                 .rename(columns={'hour':'year_peak_hour','mean':'year_peak_mean'}))
    peak_count=int((yearly_peak['year_peak_hour']==ref_hour).sum())

    reference=pd.DataFrame([{'reference_hour':ref_hour,'pooled_mean_c':selected['mean'],
        'pooled_median_c':selected['median'],'pooled_p95_c':selected['p95'],
        'year_coverage':int(selected['year_coverage']),'yearly_peak_match_count':peak_count,
        'method':'2021~2025 서울 ASOS 108, 6~8월 시간대별 평균 지면온도 최대(동률 시 p95 우선)'}])
    save_table(reference,'reference_time.csv')

    print('\n[시간대별 여름철 지면온도-상위시간]')
    display(eligible.sort_values('mean',ascending=False).head(10))
    print('\n[연도별 평균 지면온도 최고 시간]')
    display(yearly_peak)
    print(f'\n✓ 자동 선정 기준시간:{ref_hour:02d}:00')
    print('이 시간은 이후 건물 그림자 계산의 기준시간으로 사용합니다.')
    REF_HOUR=ref_hour
    mark_done('asos')
else:
    print('기존 결과 재사용: asos')
    ref=pd.read_csv(OUT/'reference_time.csv')
    REF_HOUR=int(ref.loc[0,'reference_hour'])
    print(f'기준시간:{REF_HOUR:02d}:00')


저장: asos_summer_clean.parquet 행: 11033
저장: asos_hourly.csv 행: 120
저장: reference_time.csv 행: 1

[시간대별 여름철 지면온도-상위시간]


,hour,n,mean,median,maximum,p95,year_coverage
13,13,459,37.176906,36.40,64.6,55.510,5
14,14,460,37.120435,36.55,64.1,55.210,5
12,12,460,36.462174,35.55,62.2,53.705,5
15,15,460,36.159565,35.30,59.7,52.000,5
11,11,460,35.005000,34.05,57.4,50.105,5
16,16,460,34.462174,34.10,54.4,48.305,5
10,10,459,32.503050,31.60,51.9,45.010,5
17,17,460,32.175435,31.65,48.4,42.105,5
9,9,460,29.509130,28.95,43.6,37.605,5
18,18,460,29.412826,29.40,40.0,36.100,5



[연도별 평균 지면온도 최고 시간]


,year,year_peak_hour,year_peak_mean
13,2021,13,34.960870
37,2022,13,34.935165
61,2023,13,37.320652
85,2024,13,40.368478
110,2025,14,38.573913



✓ 자동 선정 기준시간:13:00
이 시간은 이후 건물 그림자 계산의 기준시간으로 사용합니다.
챕터 실행 종료: asos


## 3장. 고령 생활인구 — 연령 컬럼 점검 후 집계

In [37]:
if 'require_setup' not in globals():
    raise RuntimeError('3장 실행 전 0장 환경설정 → 1~2장을 순서대로 실행하세요.')
require_setup(['gpd','OUT','YEARS','input_file','read_csv_auto','save_table','REF_HOUR'], '3장')
if not (OUT/'districts.gpkg').exists():
    raise FileNotFoundError('1장 districts.gpkg를 먼저 생성하세요.')

# ============================================================
# 3장. 고령인구
# 등록인구 65+ = 5개년 공통 주 지표
# 생활인구 = 시기별 정의를 보존한 보조 지표
# 남/여/합계 모두 보존
# ============================================================

districts=gpd.read_file(OUT/'districts.gpkg')
districts['district_code']=districts['district_code'].astype(str).str.replace(r'\.0$','',regex=True).str.zfill(5)
SEOUL_25=set(districts['district_name'].astype(str).str.strip())

# ------------------------------------------------------------
# 3-1. 생활인구 연령구조 확인
# ------------------------------------------------------------
print('=== 생활인구 연령구조 ===')

for year in YEARS:
    p=input_file(f'elderly_{year}')
    d=read_csv_auto(p,nrows=2)

    age_cols=[
        c for c in d.columns
        if any(k in norm(c).lower()
               for k in ['65','70','75','80','85','90','95','100'])
    ]

    print(year,age_cols)

# ------------------------------------------------------------
# 3-2. 생활인구
#
# 2021~2023:
# 65~69 + 70세 이상 → 65+ 계산 가능
#
# 2024~2025:
# 65~69 + 70~74 → 65~74만 계산 가능
#
# 서로 동일 정의가 아니므로 definition을 반드시 보존
# ------------------------------------------------------------

LIVING_COLS={
    2021:{
        'male':['남자65세부터69세생활인구수','남자70세이상생활인구수'],
        'female':['여자65세부터69세생활인구수','여자70세이상생활인구수'],
        'definition':'65+'
    },
    2022:{
        'male':['남자65세부터69세생활인구수','남자70세이상생활인구수'],
        'female':['여자65세부터69세생활인구수','여자70세이상생활인구수'],
        'definition':'65+'
    },
    2023:{
        'male':['남자65세부터69세생활인구수','남자70세이상생활인구수'],
        'female':['여자65세부터69세생활인구수','여자70세이상생활인구수'],
        'definition':'65+'
    },
    2024:{
        'male':['male_f65t69_lvpop_co','male_f70t74_lvpop_co'],
        'female':['female_f65t69_lvpop_co','female_f70t74_lvpop_co'],
        'definition':'65-74'
    },
    2025:{
        'male':['male_f65t69_lvpop_co','male_f70t74_lvpop_co'],
        'female':['female_f65t69_lvpop_co','female_f70t74_lvpop_co'],
        'definition':'65-74'
    }
}

living_parts=[]

for year in YEARS:
    p=input_file(f'elderly_{year}')
    enc='cp949' if year<=2023 else 'utf-8-sig'

    for chunk in pd.read_csv(p,encoding=enc,chunksize=150000,low_memory=False):

        datecol=next((c for c in chunk if norm(c).lower() in ['stdr_de_id','기준일id']),None)
        hourcol=next((c for c in chunk if norm(c).lower() in ['tmzon_pd_se','시간대구분']),None)
        gucol=next((c for c in chunk if norm(c).lower() in ['adstrd_code_se','자치구코드']),None)

        if None in (datecol,hourcol,gucol):
            raise ValueError(f'{year}:필수 컬럼 없음')

        male_cols=LIVING_COLS[year]['male']
        female_cols=LIVING_COLS[year]['female']

        missing=set(male_cols+female_cols)-set(chunk.columns)

        if missing:
            raise ValueError(f'{year}:연령 컬럼 없음:{missing}')

        dt=pd.to_datetime(
            chunk[datecol].astype(str).str.replace(r'\.0$','',regex=True),
            format='%Y%m%d',
            errors='coerce'
        )

        hour=pd.to_numeric(chunk[hourcol],errors='coerce')

        mask=dt.dt.month.between(6,8)&(hour==REF_HOUR)

        sub=chunk.loc[mask].copy()

        sub['living_male_elderly']=(
            sub[male_cols]
            .apply(pd.to_numeric,errors='coerce')
            .sum(axis=1,min_count=len(male_cols))
        )

        sub['living_female_elderly']=(
            sub[female_cols]
            .apply(pd.to_numeric,errors='coerce')
            .sum(axis=1,min_count=len(female_cols))
        )

        sub['living_total_elderly']=(
            sub['living_male_elderly']
            +sub['living_female_elderly']
        )

        sub['district_code']=(
            sub[gucol].astype(str)
            .str.replace(r'\.0$','',regex=True)
            .str.zfill(5)
        )

        sub['date']=dt.loc[sub.index]
        sub['year']=year
        sub['living_age_definition']=LIVING_COLS[year]['definition']

        living_parts.append(
            sub[
                ['year','date','district_code',
                 'living_age_definition',
                 'living_male_elderly',
                 'living_female_elderly',
                 'living_total_elderly']
            ]
        )

living=pd.concat(living_parts,ignore_index=True)

# 같은 날·자치구 내 행 합계
living_daily=(
    living.groupby(
        ['year','date','district_code','living_age_definition'],
        as_index=False
    )[
        ['living_male_elderly',
         'living_female_elderly',
         'living_total_elderly']
    ].sum()
)

# 여름철 13시 평균
living_gu=(
    living_daily.groupby(
        ['year','district_code','living_age_definition'],
        as_index=False
    )[
        ['living_male_elderly',
         'living_female_elderly',
         'living_total_elderly']
    ].mean()
)

living_gu['living_male_ratio']=(
    living_gu.living_male_elderly/
    living_gu.living_total_elderly
)

living_gu['living_female_ratio']=(
    living_gu.living_female_elderly/
    living_gu.living_total_elderly
)



=== 생활인구 연령구조 ===
2021 ['남자65세부터69세생활인구수', '남자70세이상생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']
2022 ['남자65세부터69세생활인구수', '남자70세이상생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']
2023 ['남자65세부터69세생활인구수', '남자70세이상생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']
2024 ['male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']
2025 ['male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']


In [38]:
# ============================================================
# 3장 마지막 - 고령 생활인구 검증 + 연도별 서울 전체 집계 + CSV 저장
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

OUT=Path('/Users/hongbini/Documents/vscode_HB/교학/data/processed')
OUT.mkdir(parents=True,exist_ok=True)

if 'living_gu' not in globals(): raise NameError("living_gu가 없습니다. 고령 생활인구 전처리 셀을 먼저 실행하세요.")

living_gu=living_gu.sort_values(['year','district_code']).reset_index(drop=True)

# ------------------------------------------------------------
# 1. 자치구별 값 검증
# ------------------------------------------------------------
check=living_gu['living_male_elderly']+living_gu['living_female_elderly']
if not np.allclose(check,living_gu['living_total_elderly'],equal_nan=True): raise ValueError('남성 + 여성 != 전체 고령 생활인구')

ratio_sum=living_gu['living_male_ratio']+living_gu['living_female_ratio']
if not np.allclose(ratio_sum.dropna(),1.0,atol=1e-8): raise ValueError('남녀 비율 합계 오류')

# ------------------------------------------------------------
# 2. 자치구별 요약
# ------------------------------------------------------------
print('\n=== 고령 생활인구 자치구별 요약 ===')

summary=living_gu.groupby(['year','living_age_definition']).agg(
    districts=('district_code','nunique'),
    district_mean=('living_total_elderly','mean'),
    male_mean=('living_male_elderly','mean'),
    female_mean=('living_female_elderly','mean')
).reset_index()

display(summary)

cols=['year','district_code','living_age_definition',
      'living_total_elderly','living_male_elderly','living_female_elderly',
      'living_male_ratio','living_female_ratio']
display(living_gu[cols].head(30))

# ------------------------------------------------------------
# 3. 연도별 서울 전체 고령 생활인구
# 자치구별 여름철 기준시각 평균값을 25개 구에 대해 합산
# ------------------------------------------------------------
year_total=living_gu.groupby(['year','living_age_definition'],as_index=False).agg(
    total_elderly=('living_total_elderly','sum'),
    male_elderly=('living_male_elderly','sum'),
    female_elderly=('living_female_elderly','sum'),
    districts=('district_code','nunique')
)

year_total['male_ratio']=year_total['male_elderly']/year_total['total_elderly']
year_total['female_ratio']=year_total['female_elderly']/year_total['total_elderly']

# 서울 전체에서도 남성+여성=전체 검증
check_total=year_total['male_elderly']+year_total['female_elderly']
if not np.allclose(check_total,year_total['total_elderly'],equal_nan=True): raise ValueError('서울 전체 남성 + 여성 != 전체 고령 생활인구')

print('\n=== 연도별 서울 전체 여름철 기준시각 고령 생활인구 ===')
display(year_total)

# ------------------------------------------------------------
# 4. 연도별 자치구 수 검증
# ------------------------------------------------------------
coverage=living_gu.groupby('year')['district_code'].nunique()

print('\n=== 연도별 자치구 수 ===')
print(coverage)

if not (coverage==25).all():
    print('\n⚠️ 일부 연도에서 서울 25개 자치구가 모두 존재하지 않습니다.')
else:
    print('\n✓ 모든 연도 서울 25개 자치구 확인')

# ------------------------------------------------------------
# 5. CSV 저장
# ------------------------------------------------------------
gu_path=OUT/'living_elderly_gu_year.csv'
total_path=OUT/'living_elderly_seoul_year.csv'

living_gu.to_csv(gu_path,index=False,encoding='utf-8-sig')
year_total.to_csv(total_path,index=False,encoding='utf-8-sig')

print('\n✓ 고령 생활인구 전처리 완료')
print('자치구별 저장:',gu_path)
print('서울 전체 저장:',total_path)
print('자치구별 행 수:',len(living_gu))
print('서울 전체 행 수:',len(year_total))

print('\n※ 해석 기준')
print('2021~2023: 65세 이상')
print('2024~2025: 65~74세')
print('각 값은 6~8월 기준시각의 평균 생활인구')
print('2023→2024는 연령 정의가 달라 직접적인 증감 비교 금지')


=== 고령 생활인구 자치구별 요약 ===


,year,living_age_definition,districts,district_mean,male_mean,female_mean
0,2021,65+,25,62604.405664,25237.926155,37366.479509
1,2022,65+,25,62466.623377,24964.201707,37502.421669
2,2023,65+,25,60973.619473,24289.458548,36684.160925
3,2024,65-74,25,60539.841163,23981.740495,36558.100668
4,2025,65-74,25,59536.278021,23367.254418,36169.023604


,year,district_code,living_age_definition,living_total_elderly,living_male_elderly,living_female_elderly,living_male_ratio,living_female_ratio
0,2021,11110,65+,52283.333495,24859.313893,27424.019601,0.475473,0.524527
1,2021,11140,65+,44354.452382,20618.864277,23735.588104,0.464866,0.535134
2,2021,11170,65+,40167.430428,16466.531223,23700.899205,0.409947,0.590053
3,2021,11200,65+,47424.333852,19435.576763,27988.757089,0.409823,0.590177
4,2021,11215,65+,46847.414846,18912.426851,27934.987995,0.403703,0.596297
5,2021,11230,65+,61926.125176,25588.415284,36337.709892,0.413209,0.586791
6,2021,11260,65+,59455.180290,23629.211285,35825.969005,0.397429,0.602571
7,2021,11290,65+,65250.936736,24278.505199,40972.431537,0.372079,0.627921
8,2021,11305,65+,59358.240511,22625.815435,36732.425076,0.381174,0.618826
9,2021,11320,65+,54375.190977,21223.540464,33151.650513,0.390317,0.609683



=== 연도별 서울 전체 여름철 기준시각 고령 생활인구 ===


,year,living_age_definition,total_elderly,male_elderly,female_elderly,districts,male_ratio,female_ratio
0,2021,65+,1.565110e+06,630948.153883,934161.987721,25,0.403133,0.596867
1,2022,65+,1.561666e+06,624105.042686,937560.541735,25,0.399641,0.600359
2,2023,65+,1.524340e+06,607236.463690,917104.023133,25,0.398360,0.601640
3,2024,65-74,1.513496e+06,599543.512383,913952.516688,25,0.396132,0.603868
4,2025,65-74,1.488407e+06,584181.360444,904225.590090,25,0.392488,0.607512



=== 연도별 자치구 수 ===
year
2021    25
2022    25
2023    25
2024    25
2025    25
Name: district_code, dtype: int64

✓ 모든 연도 서울 25개 자치구 확인

✓ 고령 생활인구 전처리 완료
자치구별 저장: /Users/hongbini/Documents/vscode_HB/교학/data/processed/living_elderly_gu_year.csv
서울 전체 저장: /Users/hongbini/Documents/vscode_HB/교학/data/processed/living_elderly_seoul_year.csv
자치구별 행 수: 125
서울 전체 행 수: 5

※ 해석 기준
2021~2023: 65세 이상
2024~2025: 65~74세
각 값은 6~8월 기준시각의 평균 생활인구
2023→2024는 연령 정의가 달라 직접적인 증감 비교 금지


## 4장. 가로수 — 연도별 자치구 총량 + 2025 공간 위치
- 2021~2025 추세: 공식 **자치구별 총수량**을 그대로 사용합니다.
- 2025 세부 후보지: 정제된 실제 좌표를 사용합니다.
- 과거 세부 위치를 500m 격자에 강제 복원하지 않습니다.
- 2/3/4m 버퍼는 실제 그늘이 아니라 가정 영향권 민감도 분석입니다.


In [39]:
# trees: 2026 위치정보의 25개 자치구 시트를 시트별로 표준화한 뒤 2025 공간분포로 사용합니다.
if should_run('trees',['trees_clean.gpkg','tree_distribution_2025_district.csv']):
    districts=gpd.read_file(OUT/'districts.gpkg')
    districts['district_code']=districts['district_code'].astype(str).str.replace(r'\.0$','',regex=True).str.zfill(5)
    SEOUL_25=['종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구','도봉구',
              '노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구','영등포구','동작구',
              '관악구','서초구','강남구','송파구','강동구']
    tree_path=input_file('tree_positions')
    book=pd.ExcelFile(tree_path)
    district_sheets=[s for s in book.sheet_names if s in SEOUL_25]
    print('전체 시트:',len(book.sheet_names),'분석대상 자치구 시트:',len(district_sheets))
    print('제외 관리기관 시트:',[s for s in book.sheet_names if s not in SEOUL_25])
    if set(district_sheets)!=set(SEOUL_25): raise ValueError(f'25개 자치구 시트 확인 필요:{sorted(set(SEOUL_25)-set(district_sheets))}')

    def detect_header(path,sheet):
        sample=pd.read_excel(path,sheet_name=sheet,header=None,nrows=20,dtype=str)
        for i,row in sample.iterrows():
            text=' '.join(row.fillna('').astype(str))
            if '경도' in text and '위도' in text:return i
        return None
    def find_col(cols,keys):
        for c in cols:
            x=norm(c).lower().replace(' ','')
            if any(norm(k).lower().replace(' ','') in x for k in keys):return c
        return None

    frames=[];audit=[]
    for sheet in district_sheets:
        h=detect_header(tree_path,sheet)
        if h is None:
            audit.append({'sheet':sheet,'status':'header_not_found','rows':0,'valid_coordinate':0});continue
        d=pd.read_excel(tree_path,sheet_name=sheet,header=h)
        d.columns=[norm(c).strip() for c in d.columns]
        loncol=find_col(d.columns,['경도','longitude','lon'])
        latcol=find_col(d.columns,['위도','latitude','lat'])
        speciescol=find_col(d.columns,['수종','수목명','수종명'])
        if loncol is None or latcol is None:
            audit.append({'sheet':sheet,'status':'coordinate_column_not_found','rows':len(d),'valid_coordinate':0});continue
        out=pd.DataFrame(index=d.index)
        out['management_agency']=sheet
        out['source_sheet']=sheet
        out['species']=d[speciescol].astype(str).str.strip() if speciescol is not None else np.nan
        out['lon']=pd.to_numeric(d[loncol],errors='coerce')
        out['lat']=pd.to_numeric(d[latcol],errors='coerce')
        out['source_row']=d.index+h+2
        ncoord=int((out['lon'].notna()&out['lat'].notna()).sum())
        audit.append({'sheet':sheet,'status':'ok','rows':len(d),'valid_coordinate':ncoord,
                      'lon_col':loncol,'lat_col':latcol,'species_col':speciescol})
        frames.append(out)
    audit_df=pd.DataFrame(audit)
    display(audit_df)
    save_table(audit_df,'tree_sheet_audit.csv')
    bad=audit_df[(audit_df['status']!='ok')|(audit_df['valid_coordinate']==0)]
    if len(bad):
        display(bad)
        raise ValueError('좌표를 읽지 못한 자치구 시트가 있습니다. tree_sheet_audit.csv를 확인하세요.')

    trees_raw=pd.concat(frames,ignore_index=True)
    trees_raw['record_status']=np.select([
        trees_raw['species'].astype(str).str.contains('공분',na=False),
        trees_raw['species'].astype(str).str.contains('그늘막',na=False),
        trees_raw['lon'].isna()|trees_raw['lat'].isna(),
        ~trees_raw['lon'].between(125,129)|~trees_raw['lat'].between(33,39)],
        ['empty_pit','artificial_shade','missing_coordinate','invalid_coordinate'],default='tree')
    save_table(trees_raw,'tree_record_audit.csv')
    print('\n[레코드 상태]');print(trees_raw['record_status'].value_counts(dropna=False))

    valid=trees_raw[trees_raw['record_status']=='tree'].copy()
    tree_points=gpd.GeoDataFrame(valid,geometry=gpd.points_from_xy(valid['lon'],valid['lat']),crs='EPSG:4326').to_crs(CRS)
    tree_points=gpd.sjoin(tree_points,districts[['district_code','district_name','geometry']],how='left',predicate='within').drop(columns='index_right',errors='ignore')
    tree_points=tree_points.rename(columns={'district_name':'spatial_district'})
    unmatched=tree_points[tree_points['district_code'].isna()].copy()
    if len(unmatched):save_geo(unmatched,'trees_unmatched_district.gpkg')
    tree_points=tree_points[tree_points['district_code'].notna()].copy()
    tree_points['district_code']=tree_points['district_code'].astype(str).str.zfill(5)

    # 시트명과 실제 위치 자치구가 다른 자료는 삭제하지 않고 감사용으로 확인합니다.
    mismatch=tree_points[tree_points['source_sheet']!=tree_points['spatial_district']].copy()
    print('서울경계 미연결:',len(unmatched),'시트명-실제자치구 불일치:',len(mismatch))
    if len(mismatch):save_table(mismatch.drop(columns='geometry'),'tree_district_mismatch_audit.csv')
    gu_check=tree_points.groupby(['district_code','spatial_district']).size().reset_index(name='tree_count')
    display(gu_check)
    if tree_points['district_code'].nunique()!=25:
        missing=sorted(set(SEOUL_25)-set(tree_points['spatial_district'].dropna()))
        raise ValueError(f'실제 좌표 기준 자치구가 {tree_points["district_code"].nunique()}개입니다. 누락:{missing}')
    save_geo(tree_points,'trees_clean.gpkg')

    # 2025 위치자료는 세부 공간분포의 기준자료로 보존합니다.
    # 자치구 추세 분석에서는 아래 좌표수를 공식 총수량 대신 사용하지 않습니다.
    counts=(tree_points.groupby(['district_code','spatial_district']).size().rename('base_coordinate_count').reset_index())
    counts['coordinate_share_seoul']=counts['base_coordinate_count']/counts['base_coordinate_count'].sum()
    save_table(counts,'tree_distribution_2025_district.csv')
    print('✓ 가로수 위치 전처리 완료 / 실제 좌표 기준 자치구:',counts['district_code'].nunique())
    print('※ 과거 격자별 위치는 생성하지 않습니다. 2025 실제 좌표는 후보지 분석에서 사용합니다.')
    mark_done('trees')
else:
    print('기존 결과 재사용: trees')


전체 시트: 27 분석대상 자치구 시트: 25
제외 관리기관 시트: ['중부공원여가센터', '서울시설공단']


,sheet,status,rows,valid_coordinate,lon_col,lat_col,species_col
0,종로구,ok,6898,6898,경도,위도,수종
1,중구,ok,7777,7777,좌표(경도),좌표(위도),수종
2,용산구,ok,7788,7788,좌표(경도),좌표(위도),수종
3,성동구,ok,6534,6534,좌표(경도),좌표(위도),수종
4,광진구,ok,6032,6032,좌표(경도),좌표(위도),수종
5,동대문구,ok,9482,9482,좌표(경도),좌표(위도),수종
6,중랑구,ok,7854,7854,좌표(경도),좌표(위도),수종
7,성북구,ok,7301,7301,좌표(경도),좌표(위도),수종
8,강북구,ok,5930,5930,좌표(경도),좌표(위도),수종
9,도봉구,ok,8679,8679,좌표(경도),좌표(위도),수종


저장: tree_sheet_audit.csv 행: 25
저장: tree_record_audit.csv 행: 274690

[레코드 상태]
record_status
tree                  268811
empty_pit               5877
missing_coordinate         1
artificial_shade           1
Name: count, dtype: int64
저장: trees_unmatched_district.gpkg 도형: 599
서울경계 미연결: 599 시트명-실제자치구 불일치: 1240
저장: tree_district_mismatch_audit.csv 행: 1240


,district_code,spatial_district,tree_count
0,11110,종로구,6771
1,11140,중구,7715
2,11170,용산구,7826
3,11200,성동구,6060
4,11215,광진구,5959
5,11230,동대문구,9333
6,11260,중랑구,7734
7,11290,성북구,7108
8,11305,강북구,5557
9,11320,도봉구,8509


저장: trees_clean.gpkg 도형: 268212
저장: tree_distribution_2025_district.csv 행: 25
✓ 가로수 위치 전처리 완료 / 실제 좌표 기준 자치구: 25
※ 과거 격자별 위치는 생성하지 않습니다. 2025 실제 좌표는 후보지 분석에서 사용합니다.
챕터 실행 종료: trees


In [40]:
# 4장 후속 단계용 결과를 메모리에 불러옵니다.
trees_path=OUT/'trees_clean.gpkg'
if not trees_path.exists():raise FileNotFoundError('trees_clean.gpkg가 없습니다. 먼저 가로수 위치 전처리를 실행하세요.')
tree_points=gpd.read_file(trees_path).to_crs(CRS)


In [41]:
# tree_allocation: 공식 2021~2025 자치구 총량을 추세용 공급지표로 구성하고 2025 위치 영향권을 별도로 저장합니다.
if should_run('tree_allocation',['tree_totals_verified.csv','tree_district_year.parquet','tree_assumed_influence_2025.gpkg','tree_supply_sensitivity.csv']):
    districts=gpd.read_file(OUT/'districts.gpkg')
    districts['district_code']=districts['district_code'].astype(str).str.zfill(5)
    SEOUL_25=['종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구','도봉구',
              '노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구','영등포구','동작구',
              '관악구','서초구','강남구','송파구','강동구']
    raw=read_csv_auto(input_file('tree_totals'),header=None)
    name_to_code=dict(zip(districts['district_name'].astype(str).str.strip(),districts['district_code']))
    year_cols={}
    for c in range(raw.shape[1]):
        y=str(raw.iloc[0,c]).strip();h2=str(raw.iloc[1,c]).strip();h3=str(raw.iloc[2,c]).strip()
        if y in ['2021','2022','2023','2024','2025'] and '수량' in h2 and h3=='소계':year_cols[int(y)]=c
    if set(year_cols)!={2021,2022,2023,2024,2025}:raise ValueError(f'연도별 수량 열 탐색 실패:{year_cols}')
    records=[]
    for r in range(3,len(raw)):
        gu=str(raw.iloc[r,1]).strip()
        if gu not in SEOUL_25:continue
        code=name_to_code.get(gu)
        if code is None:raise ValueError(f'{gu} 자치구 코드 없음')
        for year,c in year_cols.items():
            v=raw.iloc[r,c]
            if pd.isna(v):n=np.nan
            else:
                v=str(v).replace(',','').strip()
                n=0 if v in ['','-','nan','None'] else pd.to_numeric(v,errors='coerce')
            records.append({'year':year,'district_name':gu,'district_code':code,'total_trees':n})
    totals=pd.DataFrame(records)
    totals['district_code']=totals['district_code'].astype(str).str.zfill(5)
    totals['total_trees']=pd.to_numeric(totals['total_trees'],errors='coerce')
    if len(totals)!=125:raise ValueError(f'25개구×5개년=125행이어야 하지만 {len(totals)}행')
    if totals.duplicated(['year','district_code']).any():raise ValueError('연도-자치구 중복 존재')
    if totals['total_trees'].isna().any():
        display(totals[totals['total_trees'].isna()]);raise ValueError('공식 가로수 수량 결측 존재')
    save_table(totals,'tree_totals_verified.csv')
    display(totals.groupby('year')['total_trees'].sum().reset_index(name='district_25_total'))

    # 자치구 연도별 공식 수량 자체를 5개년 추세용 공급지표로 사용합니다.
    tree_gu_year=totals.copy()
    tree_gu_year['assumed_tree_supply_index']=tree_gu_year['total_trees']*np.pi*TREE_RADIUS_M**2
    tree_gu_year['index_radius_m']=TREE_RADIUS_M
    save_table(tree_gu_year,'tree_district_year.parquet')

    # 반경 가정 민감도: 실제 그늘면적이 아니라 가정 영향권 지수입니다.
    sens=[]
    for radius in TREE_RADIUS_SENSITIVITY_M:
        q=totals.copy();q['assumed_radius_m']=radius
        q['assumed_tree_supply_index']=q['total_trees']*np.pi*radius**2
        sens.append(q)
    save_table(pd.concat(sens,ignore_index=True),'tree_supply_sensitivity.csv')

    # 2025 실제 좌표에 중심 가정반경을 적용한 '영향권' 지도. 실제 태양그늘로 해석하지 않습니다.
    tree_presence=tree_points[['district_code','spatial_district','geometry']].copy()
    tree_presence['assumed_radius_m']=TREE_RADIUS_M
    tree_presence['geometry']=tree_presence.geometry.buffer(TREE_RADIUS_M)
    save_geo(tree_presence,'tree_assumed_influence_2025.gpkg')
    print('✓ 가로수 전처리 완료: 연도별 자치구 총량 + 2025 실제 위치 분리')
    print('※ assumed_tree_supply_index는 실제 그늘면적/피복률이 아닙니다.')
    mark_done('tree_allocation')
else:
    print('기존 결과 재사용: tree_allocation')


저장: tree_totals_verified.csv 행: 125


,year,district_25_total
0,2021,281285
1,2022,282921
2,2023,282041
3,2024,276699
4,2025,273837


저장: tree_district_year.parquet 행: 125
저장: tree_supply_sensitivity.csv 행: 375
저장: tree_assumed_influence_2025.gpkg 도형: 268212
✓ 가로수 전처리 완료: 연도별 자치구 총량 + 2025 실제 위치 분리
※ assumed_tree_supply_index는 실제 그늘면적/피복률이 아닙니다.
챕터 실행 종료: tree_allocation


## 5장. 건물 위치·높이·그림자
### 5-0. 공간기반자료 재로딩 및 검증 설정
0장을 먼저 실행합니다. **5-2~5-8이 높이 전처리**, 5-9 이후는 별도 선택 실행입니다.

2026 자료의 높이는 겹침·승인일만으로 자동 확정하지 않습니다. 후보표를 검토한 뒤 `building_identity_review.csv`의 `identity_verified`에 true, `review_note`에 검토 근거를 기록하세요. 신축·재건축·증축 가능성도 확인해야 합니다. 매칭 임계값/경계 버퍼는 연구 설정이며 검증된 사실이 아닙니다.


In [42]:
if 'require_setup' not in globals():
    raise RuntimeError('0장 환경설정부터 실행하세요.')
require_setup(['OUT','CRS','YEARS','input_file','save_geo','save_table'], '5-0')
from shapely import union_all, make_valid
from shapely.geometry import Polygon, MultiPolygon
from shapely import affinity
import math

districts=gpd.read_file(OUT/'districts.gpkg').to_crs(CRS)
SEOUL=union_all(districts.geometry)
BUILDING_BUFFER_M=500.0  # 서울 외부 후보 범위. 최대 그림자 길이보다 작으면 외부 영향이 누락됩니다.
BUILDING_DOMAIN=SEOUL.buffer(BUILDING_BUFFER_M)
MATCH_IOU_MIN=0.80  # 연구 설정값: 표본 수동검증 후 확정
MATCH_OVERLAP_MIN=0.90  # 연구 설정값: 표본 수동검증 후 확정
MATCH_SECOND_IOU_MAX=0.20  # 다중 후보 보류 기준
RUN_BUILDING_SHADOWS=False
# 실제 정의서·컬럼 출력 확인 후 설정. A열 의미는 사용자 제공 정의에 근거합니다.
GIS_FIELDS={'area':'A12','approval':'A13','height':'A16','floors':'A26',
            'id':None,'use':None}
# GIS/OSM 용도를 같은 공통 용도로 매핑. 미설정은 전체 중앙값 사용.
GIS_USE_MAP={}
OSM_USE_MAP={}
MIN_FLOOR_SAMPLE=30
# 표본 제외용 가정 설정. 분포를 확인해 채택 범위를 조정하세요.
FLOOR_HEIGHT_RANGE=(2.0,8.0)
REVIEW_PATH=OUT/'building_identity_review.csv'
REGISTRY_REVIEW_PATH=OUT/'building_registry_verified.csv'

def positive(s):
    v=pd.to_numeric(s,errors='coerce')
    return v.where(np.isfinite(v)&(v>0))

def polygons(g):
    if g is None or g.is_empty:return None
    g=make_valid(g)
    if isinstance(g,(Polygon,MultiPolygon)):return g
    parts=[]
    for x in getattr(g,'geoms',[]):
        q=polygons(x)
        if q is not None:parts.append(q)
    return union_all(parts) if parts else None

def clean_polygons(g):
    g=g.copy()
    g.geometry=g.geometry.map(polygons)
    return g[g.geometry.notna() & ~g.geometry.is_empty].copy()

def dates(s):
    return pd.to_datetime(s.astype(str).str.replace(r'\.0$','',regex=True)
        .str.replace(r'[^0-9]','',regex=True).where(lambda v:v.str.len()==8),format='%Y%m%d',errors='coerce')

def truth(s):
    return s.astype(str).str.strip().str.lower().isin(['true','1','yes','y'])

def parse_height(value):
    m=re.fullmatch(r"\s*(\d+(?:\.\d+)?)\s*(m|meter|meters|metre|metres|ft|feet|')?\s*",str(value).lower())
    if not m:return np.nan
    v=float(m.group(1))*(0.3048 if m.group(2) in ('ft','feet',"'") else 1)
    return v if v>0 and np.isfinite(v) else np.nan
print('GIS 컬럼 매핑:',GIS_FIELDS)
print('설정 변경 시 해당 building_* 단계를 FORCE_RUN에 지정하세요.')


GIS 컬럼 매핑: {'area': 'A12', 'approval': 'A13', 'height': 'A16', 'floors': 'A26', 'id': None, 'use': None}
설정 변경 시 해당 building_* 단계를 FORCE_RUN에 지정하세요.


### 5-1. 연도별 OSM 원본 점검

In [43]:
display(pd.DataFrame([{'year':y,'pbf':str(input_file(f'osm_{y}'))} for y in YEARS]))


,year,pbf
0,2021,/Users/hongbini/Documents/vscode_HB/교학/.cache/...
1,2022,/Users/hongbini/Documents/vscode_HB/교학/.cache/...
2,2023,/Users/hongbini/Documents/vscode_HB/교학/.cache/...
3,2024,/Users/hongbini/Documents/vscode_HB/교학/.cache/...
4,2025,/Users/hongbini/Documents/vscode_HB/교학/.cache/...


### 5-2. 연도별 외곽선·원래 높이/층수 추출
기존 `buildings_YYYY.gpkg`는 층수 고정 추정/영역 차이가 있어 자동 재사용하지 않습니다. 새 이름으로 한 번 추출합니다. 외곽선은 서울 경계에서 자르지 않습니다.

In [44]:
for year in YEARS:
    name=f'buildings_osm_{year}.gpkg'
    if not should_run('building_osm',[name]):continue
    try:
        from pyrosm import OSM
    except ImportError as exc:
        raise ImportError('로컬 환경에 pyrosm 설치가 필요합니다.') from exc
    domain=gpd.GeoSeries([BUILDING_DOMAIN],crs=CRS).to_crs(4326)
    b=OSM(str(input_file(f'osm_{year}')),bounding_box=domain.total_bounds.tolist()).get_buildings()
    if b is None or b.empty:raise ValueError(f'{year}: OSM 건물 없음')
    if b.crs is None:raise ValueError('OSM CRS 확인 필요')
    b=clean_polygons(b.to_crs(CRS))
    b=b[b.intersects(BUILDING_DOMAIN)].copy().reset_index(drop=True)
    if 'id' not in b:raise ValueError('OSM 원본 ID 없음')
    typ=b['osm_type'].astype(str) if 'osm_type' in b else pd.Series('feature',index=b.index)
    b['building_id']=str(year)+':'+typ+':'+b['id'].astype(str)
    # 동일 원본 ID가 중복이면 임의의 레코드를 선택하지 않습니다.
    if b.building_id.duplicated().any():raise ValueError('OSM ID 중복: 원본 유형/파트 구분 확인 필요')
    b['year']=year
    b['osm_id']=b['id'].astype(str)
    b['height_raw']=b['height'].astype(str) if 'height' in b else ''
    b['osm_height_m']=b['height_raw'].map(parse_height)
    lc=next((c for c in ['building:levels','building_levels'] if c in b),None)
    b['osm_levels']=positive(b[lc]) if lc else np.nan
    b['building']=b['building'].fillna('unknown').astype(str) if 'building' in b else 'unknown'
    b['geometry_duplicate']=b.geometry.to_wkb().duplicated(keep=False)
    b['footprint_area_m2']=b.area
    save_geo(b[['year','building_id','osm_id','building','height_raw','osm_height_m','osm_levels',
                'geometry_duplicate','footprint_area_m2','geometry']],name)
save_table(pd.DataFrame([{'year':y,'file':f'buildings_osm_{y}.gpkg',
    'exists':(OUT/f'buildings_osm_{y}.gpkg').exists()} for y in YEARS]),'building_extraction_audit.csv')


저장: buildings_osm_2021.gpkg 도형: 74304
저장: buildings_osm_2022.gpkg 도형: 86178
저장: buildings_osm_2023.gpkg 도형: 104186
저장: buildings_osm_2024.gpkg 도형: 128211
저장: buildings_osm_2025.gpkg 도형: 138074
저장: building_extraction_audit.csv 행: 5


### 5-3. 2026 GIS 전처리
695,753개는 사용자 제시 원본 수이며 실행 결과와 비교할 참고값입니다. SHP와 동일한 basename의 DBF가 필요합니다. 없으면 원본을 수정하지 않고 임시 작업 폴더에 구성합니다. 용도 필드는 정의 확인 후 `GIS_FIELDS`에 지정하세요.

In [45]:
gis_name='buildings_gis_2026_clean.gpkg'
if should_run('building_gis_clean',[gis_name]):
    shp=input_file('gis_shape').resolve()
    if not shp.with_suffix('.dbf').exists():
        import tempfile
        work=OUT/'building_gis_source_copy'
        work.mkdir(exist_ok=True)
        for ext in ['.shp','.shx','.prj','.cpg']:
            q=shp.with_suffix(ext)
            if q.exists():shutil.copy2(q,work/q.name)
        with zipfile.ZipFile(input_file('gis_dbf_zip')) as z:
            members=[m for m in z.namelist() if Path(m).suffix.lower()=='.dbf' and not m.startswith('__MACOSX/')]
            if len(members)!=1:raise ValueError('DBF 압축 내 대상 파일을 확인하세요.')
            with z.open(members[0]) as src, open(work/(shp.stem+'.dbf'),'wb') as dst:
                shutil.copyfileobj(src,dst)
        shp=work/shp.name
    gis=gpd.read_file(shp)
    print('GIS 원본 행:',len(gis),'컬럼:',gis.columns.tolist())
    required=[GIS_FIELDS[k] for k in ['area','approval','height','floors']]
    if any(c not in gis for c in required):raise ValueError('GIS_FIELDS를 실제 필드 정의에 맞게 수정하세요.')
    if gis.crs is None:raise ValueError('GIS 원본 CRS 확인 필요; 임의 지정 금지')
    gis=clean_polygons(gis.to_crs(CRS)).reset_index(drop=True)
    gis['gis_id']=gis[GIS_FIELDS['id']].astype(str) if GIS_FIELDS['id'] else 'gisrow:'+gis.index.astype(str)
    if gis.gis_id.duplicated().any():raise ValueError('GIS 식별자 중복')
    gis['area_m2']=positive(gis[GIS_FIELDS['area']])
    gis['geometry_area_m2']=gis.area
    gis['approval_date']=dates(gis[GIS_FIELDS['approval']])
    gis['height_m']=positive(gis[GIS_FIELDS['height']])
    gis['ground_floors']=positive(gis[GIS_FIELDS['floors']])
    gis['use_raw']=gis[GIS_FIELDS['use']].astype(str) if GIS_FIELDS['use'] else 'unknown'
    gis['geometry_duplicate']=gis.geometry.to_wkb().duplicated(keep=False)
    save_geo(gis[['gis_id','area_m2','geometry_area_m2','approval_date','height_m','ground_floors',
                  'use_raw','geometry_duplicate','geometry']],gis_name)
gis=gpd.read_file(OUT/gis_name).to_crs(CRS)
gis['approval_date']=pd.to_datetime(gis['approval_date'],errors='coerce')
print('GIS 정제 행:',len(gis),'중복 도형:',int(gis.geometry_duplicate.sum()))


GIS 원본 행: 695753 컬럼: ['A0', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'geometry']
저장: buildings_gis_2026_clean.gpkg 도형: 695753
GIS 정제 행: 695753 중복 도형: 707


### 5-4. OSM ↔ GIS 공간 후보 생성 및 동일성 검토표
각 OSM의 모든 교차 후보를 기록합니다. best/second IoU, 양방향 중복 매칭, 승인일을 점검하고 최종적으로 사람이 동일성 검토를 승인해야 합니다. 승인일은 각 PBF의 1월 1일 이전으로 보수적으로 제한합니다. 겹침과 날짜만으로 증축 이력이 검증되지는 않습니다.

In [46]:
all_candidates=[]
for year in YEARS:
    name=f'building_gis_candidates_{year}.parquet'
    if should_run('building_match',[name]):
        b=gpd.read_file(OUT/f'buildings_osm_{year}.gpkg')
        rows=[]
        for r in b.itertuples():
            for j in gis.sindex.query(r.geometry,predicate='intersects'):
                g=gis.iloc[j]; a=r.geometry.intersection(g.geometry).area
                if a<=0:continue
                rows.append({'year':year,'building_id':r.building_id,'gis_id':g.gis_id,
                    'gis_match_overlap':a/r.geometry.area,
                    'gis_match_iou':a/(r.geometry.area+g.geometry.area-a)})
        cand=pd.DataFrame(rows,columns=['year','building_id','gis_id','gis_match_overlap','gis_match_iou'])
        save_table(cand,name)
    all_candidates.append(pd.read_parquet(OUT/name))
candidates=pd.concat(all_candidates,ignore_index=True)
best_rows=[]
for (year,bid),group in candidates.groupby(['year','building_id']):
    g=group.sort_values(['gis_match_iou','gis_id'],ascending=[False,True])
    row=g.iloc[0].to_dict()
    row['second_iou']=g.iloc[1].gis_match_iou if len(g)>1 else 0.0
    best_rows.append(row)
best=pd.DataFrame(best_rows,columns=list(candidates.columns)+['second_iou'])
best['gis_reused']=best.duplicated(['year','gis_id'],keep=False)
attrs=gis.drop(columns='geometry').rename(columns={'height_m':'gis2026_height_m','ground_floors':'gis2026_floors',
    'approval_date':'gis_approval_date','use_raw':'gis_use_raw'})
best=best.merge(attrs,on='gis_id',how='left',validate='many_to_one')
cutoff=pd.to_datetime(best['year'].astype(str)+'-01-01')
best['geometric_date_candidate']=(best.gis_match_iou.ge(MATCH_IOU_MIN)
    & best.gis_match_overlap.ge(MATCH_OVERLAP_MIN)&best.second_iou.le(MATCH_SECOND_IOU_MAX)
    & ~best.gis_reused & ~best.geometry_duplicate.fillna(True).astype(bool)
    & best.gis_approval_date.notna() & best.gis_approval_date.le(cutoff))
save_table(best,'building_gis_best_matches.parquet')
# 신규 후보 템플릿은 별도 저장. 기존 검토를 덮어쓰지 않습니다.
review=best[['year','building_id','gis_id','gis_match_iou','gis_match_overlap','gis_approval_date','geometric_date_candidate']].copy()
review['identity_verified']=False
review['review_note']=''
save_table(review,'building_identity_review_template.csv')
if not REVIEW_PATH.exists():save_table(review,REVIEW_PATH.name)
print('building_identity_review.csvをレビュー後、5-7を実行。未承認のGIS値は採用しません。')


저장: building_gis_candidates_2021.parquet 행: 102276
저장: building_gis_candidates_2022.parquet 행: 117902
저장: building_gis_candidates_2023.parquet 행: 141997
저장: building_gis_candidates_2024.parquet 행: 176344
저장: building_gis_candidates_2025.parquet 행: 189600
저장: building_gis_best_matches.parquet 행: 485482
저장: building_identity_review_template.csv 행: 485482
저장: building_identity_review.csv 행: 485482
building_identity_review.csvをレビュー後、5-7を実行。未承認のGIS値は採用しません。


### 5-5. 건축물대장 추가 검증 입력
주소만으로 자동 결합하지 않습니다. 원본 필드를 확인하고 동명·지번·면적·승인일 및 변경 이력을 비교한 **건물별 검증값**만 아래 표에 입력하세요. GIS 후보 없음/불명확 건물도 직접 검토할 수 있습니다. 높이·층수는 양수, 승인일은 PBF 시점 이전이어야 합니다.

In [47]:
registry=input_file('building_register')
print('건축물대장 원본:',registry)
display(read_csv_auto(registry,nrows=5))
cols=['year','building_id','registry_height_m','registry_floors','registry_approval_date',
      'registry_use','identity_verified','registry_record_id','review_note']
if not REGISTRY_REVIEW_PATH.exists():
    save_table(pd.DataFrame(columns=cols),REGISTRY_REVIEW_PATH.name)
print('검증 입력:',REGISTRY_REVIEW_PATH)


건축물대장 원본: /Users/hongbini/Documents/vscode_HB/교학/.cache/flat_inputs/서울시 건축물대장 표제부.csv


,대지위치,시군구코드명,법정동코드명,대지구분코드명,주지번,부지번,특수지명,블록번호,로트번호,새주소도로코드명,...,에너지효율등급값,에너지절감률,EPI점수,친환경건축물등급값,친환경건축물인증점수,지능형건축물등급값,지능형건축물인증점수,내진설계적용여부,내진능력내용,데이터수집일
0,서울특별시 강서구 방화동 246-66,서울특별시 강서구,방화동,대지,246,66,NaN,NaN,NaN,방화대로,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,2024-11-19 14:44:57.338
1,서울특별시 마포구 아현동 751-11,서울특별시 마포구,아현동,대지,751,11,NaN,NaN,NaN,환일길,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,2024-11-19 14:44:57.338
2,서울특별시 양천구 신월동 987-1,서울특별시 양천구,신월동,대지,987,1,NaN,NaN,NaN,신월로,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,2024-11-19 14:44:57.338
3,서울특별시 양천구 신정동 87-39,서울특별시 양천구,신정동,대지,87,39,NaN,NaN,NaN,신목로,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,2024-11-19 14:44:57.338
4,서울특별시 강서구 화곡동 102-29,서울특별시 강서구,화곡동,대지,102,29,NaN,NaN,NaN,까치산로4길,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,2024-11-19 14:44:57.338


저장: building_registry_verified.csv 행: 0
검증 입력: /Users/hongbini/Documents/vscode_HB/교학/data/processed/building_registry_verified.csv


### 5-6. 실제 GIS에서 용도별 층당높이 중앙값 산출
용도 매핑이 없거나 표본 수가 적으면 전체 적격 표본 중앙값으로 대체하고 출처를 기록합니다. 표본 범위·수 기준은 민감도 검토 대상입니다.

In [48]:
sample=gis[~gis.geometry_duplicate.astype(bool)].copy()
sample['floor_height_m']=sample.height_m/sample.ground_floors
sample=sample[sample.floor_height_m.between(*FLOOR_HEIGHT_RANGE)].copy()
sample['use_group']=sample.use_raw.map(GIS_USE_MAP).fillna('unknown')
if sample.empty:raise ValueError('적격 층당높이 표본 없음: 원본/제외 범위를 확인하세요.')
medians=sample.groupby('use_group').floor_height_m.agg(['count','median']).reset_index()
medians.loc[len(medians)]=['__all__',len(sample),sample.floor_height_m.median()]
save_table(medians,'building_floor_height_medians.csv')
display(medians)


저장: building_floor_height_medians.csv 행: 2


/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2649603841.py:4: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  sample['use_group']=sample.use_raw.map(GIS_USE_MAP).fillna('unknown')


,use_group,count,median
0,unknown,303749,3.25
1,__all__,303749,3.25


### 5-7. 최종 높이 결정
우선순위: OSM 높이 → 동일성 검토된 GIS 높이 → 검증된 대장 높이 → 층수×실측 기반 용도별 중앙값 → unknown. GIS/대장 층수와 용도 역시 동일성 승인된 경우에만 이용합니다. 미확인 건물도 삭제하지 않습니다.

In [49]:
best=pd.read_parquet(OUT/'building_gis_best_matches.parquet')
rev=read_csv_auto(REVIEW_PATH)
keys=['year','building_id','gis_id']
if rev.duplicated(keys).any():raise ValueError('동일성 검토표 키 중복')
rev['review_ok']=truth(rev.identity_verified)&rev.review_note.fillna('').str.strip().ne('')
best=best.merge(rev[keys+['review_ok']],on=keys,how='left',validate='one_to_one')
best['gis_verified']=best.geometric_date_candidate & best.review_ok.fillna(False)
reg=read_csv_auto(REGISTRY_REVIEW_PATH)
if reg.duplicated(['year','building_id']).any():raise ValueError('대장 검증표 키 중복')
reg['registry_approval_date']=dates(reg.registry_approval_date)
reg['registry_ok']=(truth(reg.identity_verified)&reg.review_note.fillna('').str.strip().ne('')
    &reg.registry_record_id.fillna('').astype(str).str.strip().ne('')
    &reg.registry_approval_date.notna()
    &reg.registry_approval_date.le(pd.to_datetime(reg.year.astype(str)+'-01-01')))
reg['registry_height_m']=positive(reg.registry_height_m)
reg['registry_floors']=positive(reg.registry_floors)
med=read_csv_auto(OUT/'building_floor_height_medians.csv')
all_median=float(med.loc[med.use_group=='__all__','median'].iloc[0])
use_median=med[(med['count']>=MIN_FLOOR_SAMPLE)&~med.use_group.isin(['unknown','__all__'])].set_index('use_group')['median']
for year in YEARS:
    b=gpd.read_file(OUT/f'buildings_osm_{year}.gpkg')
    # Keep OSM duplication flag distinct from GIS flag.
    b=b.rename(columns={'geometry_duplicate':'osm_geometry_duplicate'})
    b=b.merge(best.drop(columns='year'),on='building_id',how='left',validate='one_to_one')
    b=b.merge(reg.drop(columns=['year','identity_verified','review_note']),on='building_id',how='left',validate='one_to_one')
    gv=b.gis_verified.fillna(False).astype(bool); rv=b.registry_ok.fillna(False).astype(bool)
    b['applied_height_m']=positive(b.osm_height_m)
    b['height_source']=np.where(b.applied_height_m.notna(),'osm_height','unknown')
    for vals,ok,label in [(b.gis2026_height_m,gv,'gis2026_verified'),(b.registry_height_m,rv,'registry_verified')]:
        mask=b.applied_height_m.isna()&ok&positive(vals).notna()
        b.loc[mask,'applied_height_m']=positive(vals)[mask]
        b.loc[mask,'height_source']=label
    floors=positive(b.osm_levels)
    floor_source=pd.Series(np.where(floors.notna(),'osm_levels','unknown'),index=b.index)
    for vals,ok,label in [(b.gis2026_floors,gv,'gis2026_verified'),(b.registry_floors,rv,'registry_verified')]:
        mask=floors.isna()&ok&positive(vals).notna()
        floors.loc[mask]=positive(vals)[mask];floor_source.loc[mask]=label
    use=b.building.map(OSM_USE_MAP).fillna('unknown')
    gu=b.gis_use_raw.map(GIS_USE_MAP)
    use=use.mask(gv&gu.notna(),gu)
    use=use.mask(rv&b.registry_use.notna(),b.registry_use)
    fh=use.map(use_median)
    b['floor_height_basis']=np.where(fh.notna(),'use_group_median','overall_median')
    b['floor_height_m']=fh.fillna(all_median)
    b['floor_source']=floor_source
    b['use_group']=use
    mask=b.applied_height_m.isna()&floors.notna()
    b.loc[mask,'applied_height_m']=floors[mask]*b.loc[mask,'floor_height_m']
    b.loc[mask,'height_source']='floor_estimated'
    b['height_is_estimated']=b.height_source.eq('floor_estimated')
    b['height_unknown']=b.applied_height_m.isna()
    save_geo(b,f'buildings_final_{year}.gpkg')


/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2184770269.py:5: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  rev['review_ok']=truth(rev.identity_verified)&rev.review_note.fillna('').str.strip().ne('')
/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2184770269.py:38: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  use=b.building.map(OSM_USE_MAP).fillna('unknown')


저장: buildings_final_2021.gpkg 도형: 74304


/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2184770269.py:38: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  use=b.building.map(OSM_USE_MAP).fillna('unknown')


저장: buildings_final_2022.gpkg 도형: 86178


/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2184770269.py:38: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  use=b.building.map(OSM_USE_MAP).fillna('unknown')


저장: buildings_final_2023.gpkg 도형: 104186


/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2184770269.py:38: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  use=b.building.map(OSM_USE_MAP).fillna('unknown')


저장: buildings_final_2024.gpkg 도형: 128211


/var/folders/f2/2m2prsw54cq18v_2h47r1b1m0000gn/T/ipykernel_99023/2184770269.py:38: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  use=b.building.map(OSM_USE_MAP).fillna('unknown')


저장: buildings_final_2025.gpkg 도형: 138074


### 5-8. 높이 확보율 — 개수·외곽선 면적
확보율은 추정 높이를 포함하며, 직접 태그/검증 높이 비율과 추정 비율을 함께 보고합니다. 분석 영역은 서울시 내부, 경계 외부 후보 건물은 별도입니다.

In [50]:
rows=[]
for year in YEARS:
    b=gpd.read_file(OUT/f'buildings_final_{year}.gpkg')
    area=b.geometry.intersection(SEOUL).area
    b=b[area>0].copy();area=area[area>0]
    known=positive(b.applied_height_m).notna()
    est=b.height_source.eq('floor_estimated')
    rows.append({'year':year,'building_count':len(b),'height_known':int(known.sum()),
        'height_unknown':int((~known).sum()),'height_coverage_pct':100*known.mean(),
        'footprint_area_m2':area.sum(),'known_footprint_m2':area[known].sum(),
        'unknown_footprint_m2':area[~known].sum(),
        'height_known_area_rate':100*area[known].sum()/area.sum() if area.sum()>0 else np.nan,
        'estimated_count':int(est.sum()),'estimated_area_rate':100*area[est].sum()/area.sum() if area.sum()>0 else np.nan})
audit=pd.DataFrame(rows);display(audit);save_table(audit,'building_height_audit.csv')
print('最終高さファイル生成≠全建物検証完了。unknown・推定率・レビュー根拠を確認。'.replace('最終高さファイル生成≠全建物検証完了。unknown・推定率・レビュー根拠を確認。','최종 파일 생성 ≠ 모든 건물 검증 완료. unknown·추정 비율·검토 근거를 확인하세요.'))


,year,building_count,height_known,height_unknown,height_coverage_pct,footprint_area_m2,known_footprint_m2,unknown_footprint_m2,height_known_area_rate,estimated_count,estimated_area_rate
0,2021,72507,13588,58919,18.740260,3.406742e+07,7.185608e+06,2.688182e+07,21.092315,10944,17.468752
1,2022,84112,15459,68653,18.379066,3.757953e+07,8.277295e+06,2.930223e+07,22.026074,12570,18.542844
2,2023,101716,16534,85182,16.255063,4.163491e+07,8.859597e+06,3.277532e+07,21.279250,13521,17.763491
3,2024,125313,17957,107356,14.329718,4.584851e+07,9.340497e+06,3.650801e+07,20.372519,14754,16.945669
4,2025,133876,19815,114061,14.801010,4.764742e+07,1.024093e+07,3.740648e+07,21.493155,15777,17.695751


저장: building_height_audit.csv 행: 5
최종 파일 생성 ≠ 모든 건물 검증 완료. unknown·추정 비율·검토 근거를 확인하세요.


### 5-9~5-12. 선택 실행 — 태양 위치·건물 그림자·자치구 집계
높이 품질검토가 끝난 뒤 실행합니다. **주 분석 집계단위는 자치구**입니다. 세부 격자 집계는 공간 LST가 준비된 후보지 분석 단계에서 별도로 수행합니다.


In [51]:
def swept_polygon(geom,dx,dy):
    parts=list(geom.geoms) if isinstance(geom,MultiPolygon) else [geom]
    result=[geom,affinity.translate(geom,xoff=dx,yoff=dy)]
    for part in parts:
        for ring in [part.exterior,*part.interiors]:
            coords=list(ring.coords)
            for a,b in zip(coords[:-1],coords[1:]):
                q=Polygon([a,b,(b[0]+dx,b[1]+dy),(a[0]+dx,a[1]+dy)])
                if q.area>0:result.append(q)
    return polygons(union_all(result))

if not RUN_BUILDING_SHADOWS:
    print('높이 품질 검토 전: 그림자·자치구 집계 생략')
else:
    import pvlib
    outputs=[];all_shadow=[];solar=[]
    for year in YEARS:
        b=gpd.read_file(OUT/f'buildings_final_{year}.gpkg').to_crs(CRS)
        ts=pd.DatetimeIndex([f'{year}-08-01 13:00'],tz='Asia/Seoul')
        pos=pvlib.solarposition.get_solarposition(ts,37.5665,126.9780).iloc[0]
        alpha=float(pos.apparent_elevation);az=float(pos.azimuth)
        if alpha<=0:raise ValueError('태양고도 <=0')
        solar.append({'year':year,'datetime_kst':str(ts[0]),'altitude_deg':alpha,'azimuth_deg':az})
        factor=1/math.tan(math.radians(alpha));direction=math.radians((az+180)%360)
        known=positive(b.applied_height_m).notna()
        if not known.any():raise ValueError(f'{year}: 유효 높이 없음')
        max_length=float(b.loc[known,'applied_height_m'].max()*factor)
        if max_length>BUILDING_BUFFER_M:
            raise ValueError(f'{year}: 최대 그림자 {max_length:.1f}m > 버퍼. 5-0 버퍼 확대 후 OSM부터 재실행하세요.')
        shapes=[]
        for r in b[known].itertuples():
            length=r.applied_height_m*factor
            shapes.append(swept_polygon(r.geometry,length*math.sin(direction),length*math.cos(direction)))
        # 높이 미확인을 포함한 전체 건물 외곽선 합집합 제거.
        shade=union_all(shapes).difference(union_all(b.geometry)).intersection(SEOUL)
        sg=gpd.GeoDataFrame({'year':[year]},geometry=[shade],crs=CRS)
        save_geo(sg,f'building_shadow_{year}.gpkg');all_shadow.append(sg)
        for r in districts.itertuples():
            geom=r.geometry;ga=geom.area
            inds=b.sindex.query(geom,predicate='intersects')
            sub=b.iloc[inds].copy();areas=sub.geometry.intersection(geom).area
            keep=areas>0;sub=sub.loc[keep].copy();areas=areas.loc[keep]
            valid=positive(sub.applied_height_m).notna()
            sa=shade.intersection(geom).area
            if sa>ga+1e-5:raise ValueError('그늘면적 > 자치구면적')
            total=areas.sum()
            outputs.append({'year':year,'district_code':r.district_code,'district_name':r.district_name,
                'district_area_m2':ga,'building_shadow_area_m2':sa,'building_shadow_rate':100*sa/ga if ga>0 else np.nan,
                'building_count':len(sub),'height_known_count':int(valid.sum()),'height_unknown_count':int((~valid).sum()),
                'height_known_rate':100*valid.mean() if len(sub) else np.nan,
                'known_building_footprint_m2':areas[valid].sum(),'unknown_building_footprint_m2':areas[~valid].sum(),
                'height_known_area_rate':100*areas[valid].sum()/total if total>0 else np.nan})
    save_table(pd.DataFrame(solar),'building_solar_positions.csv')
    save_geo(gpd.GeoDataFrame(pd.concat(all_shadow,ignore_index=True),crs=CRS),'building_shadow_year.gpkg')
    save_table(pd.DataFrame(outputs),'building_shade_district_year.parquet')


높이 품질 검토 전: 그림자·자치구 집계 생략


## 7장. 버스정류소 — 5개년 스냅샷

In [52]:
# ============================================================
# 7장. 버스정류소 + 스마트쉼터 전처리
# ============================================================
# 주의:
# - 최초 수정 실행 시 FORCE_RUN = {'bus'} 로 설정
# - confirmed = 스마트쉼터 시설 존재 확인
# - confirmed ≠ 실제 그늘 존재/그늘 면적
# - unknown ≠ 그늘 없음
# ============================================================


# ============================================================
# 7-1. 2021~2025 버스정류장 전처리
# ============================================================

if should_run('bus', ['bus_stops_clean.gpkg','bus_stops_year.gpkg']):

    districts = gpd.read_file(OUT/'districts.gpkg').to_crs(CRS)
    bus_parts = []

    for year in YEARS:
        p = input_file(f'bus_{year}')
        d = read_csv_auto(p) if p.suffix.lower()=='.csv' else pd.read_excel(p)

        print(f'\n[{year}] 원본 컬럼:', d.columns.tolist())
        print(f'[{year}] 원본 행:', len(d))

        rename = {
            'STDR_DE':'snapshot_date','NODE_ID':'node_id',
            'STTN_NO':'ars_id','ARS_ID':'ars_id',
            'STTN_NM':'stop_name','정류소명':'stop_name','정류장명':'stop_name',
            'CRDNT_X':'lon','CRDNT_Y':'lat','X좌표':'lon','Y좌표':'lat',
            '경도':'lon','위도':'lat',
            'STTN_TY':'stop_type','정류소타입':'stop_type','정류장타입':'stop_type'
        }
        d = d.rename(columns=rename)

        if 'lon' not in d: raise ValueError(f'{year}: lon 컬럼 없음')
        if 'lat' not in d: raise ValueError(f'{year}: lat 컬럼 없음')

        if 'snapshot_date' not in d:
            stamp = re.search(r'(20\d{6})', norm(p.name))
            d['snapshot_date'] = stamp.group(1) if stamp else pd.NA

        for c in ['node_id','ars_id','stop_name','stop_type']:
            if c not in d: d[c] = pd.NA

        d['year'] = year

        for c in ['node_id','ars_id']:
            d[c] = d[c].astype('string').str.replace(r'\.0$','',regex=True).str.strip()
            d.loc[d[c].isin(['','nan','None','<NA>']), c] = pd.NA

        for c in ['stop_name','stop_type']:
            d[c] = d[c].astype('string').str.strip()
            d.loc[d[c].isin(['','nan','None','<NA>']), c] = pd.NA

        d['snapshot_date'] = pd.to_datetime(
            d['snapshot_date'].astype('string').str.replace(r'\.0$','',regex=True).str.strip(),
            format='%Y%m%d', errors='coerce'
        )

        d['lon'] = pd.to_numeric(d['lon'], errors='coerce')
        d['lat'] = pd.to_numeric(d['lat'], errors='coerce')
        d['coordinate_valid'] = d['lon'].between(125,129) & d['lat'].between(33,39)

        print(f'[{year}] 좌표 유효:', d['coordinate_valid'].sum())
        print(f'[{year}] 좌표 비정상:', (~d['coordinate_valid']).sum())

        d = d[d['coordinate_valid']].copy()

        # NODE → ARS → 이름+좌표 순으로 통합 ID 생성
        d['stop_id_source'] = np.select(
            [d['node_id'].notna(), d['ars_id'].notna()],
            ['node_id','ars_id'], default='name_coordinate'
        )

        name_key = d['stop_name'].fillna('UNKNOWN').astype('string').str.strip()
        lon_key = d['lon'].round(6).astype('string')
        lat_key = d['lat'].round(6).astype('string')

        d['stop_id'] = np.where(
            d['node_id'].notna(), 'NODE_'+d['node_id'].fillna(''),
            np.where(
                d['ars_id'].notna(), 'ARS_'+d['ars_id'].fillna(''),
                'LOC_'+name_key+'_'+lon_key+'_'+lat_key
            )
        )

        before = len(d)
        d = d.sort_values(['snapshot_date','stop_id']).drop_duplicates(
            ['snapshot_date','stop_id'], keep='last'
        ).copy()
        print(f'[{year}] 중복 제거:', before-len(d))

        bus_parts.append(d[
            ['year','snapshot_date','stop_id','stop_id_source','node_id','ars_id',
             'stop_name','stop_type','lon','lat','coordinate_valid']
        ].copy())

    if not bus_parts: raise ValueError('버스정류장 데이터를 생성하지 못했습니다.')

    bus = pd.concat(bus_parts, ignore_index=True)
    bus_geo = gpd.GeoDataFrame(
        bus, geometry=gpd.points_from_xy(bus.lon,bus.lat), crs='EPSG:4326'
    ).to_crs(CRS)

    save_geo(bus_geo, 'bus_stops_clean.gpkg')
    bus_geo['month'] = bus_geo['snapshot_date'].dt.month

    # --------------------------------------------------------
    # 연도별 대표 스냅샷
    # 여름자료 존재 → 가장 늦은 여름 스냅샷
    # 여름자료 없음 → 가장 늦은 이용 가능 스냅샷
    # --------------------------------------------------------

    representative_parts, snapshot_audit = [], []

    for year in YEARS:
        y = bus_geo[bus_geo['year'].eq(year)].copy()
        if y.empty:
            print(f'⚠️ {year}: 자료 없음')
            continue

        dated = y[y['snapshot_date'].notna()].copy()

        if not dated.empty:
            summer_dates = dated.loc[
                dated['month'].isin([6,7,8]), 'snapshot_date'
            ].dropna().drop_duplicates().sort_values()

            if len(summer_dates):
                selected_date = summer_dates.iloc[-1]
                rule = 'latest_summer_snapshot'
            else:
                selected_date = dated['snapshot_date'].max()
                rule = 'latest_available_snapshot'

            selected = y[y['snapshot_date'].eq(selected_date)].copy()
        else:
            selected_date, rule, selected = pd.NaT, 'source_date_unknown', y.copy()

        if selected['stop_id'].duplicated().sum():
            selected = selected.drop_duplicates('stop_id', keep='last').copy()

        selected['snapshot_selection_rule'] = rule
        representative_parts.append(selected)
        snapshot_audit.append({
            'year':year, 'selected_snapshot_date':selected_date,
            'selection_rule':rule, 'stop_count':len(selected)
        })

    representative = pd.concat(representative_parts, ignore_index=True)

    # 서울 자치구 공간조인
    representative = gpd.sjoin(
        representative,
        districts[['district_code','district_name','geometry']],
        how='left', predicate='within'
    ).drop(columns='index_right', errors='ignore')

    representative['within_seoul'] = representative['district_code'].notna()
    outside = representative[~representative['within_seoul']].copy()

    print('\n[서울 자치구 공간조인]')
    print('서울 내부:', representative['within_seoul'].sum())
    print('서울 경계 연결 실패:', len(outside))

    if len(outside): save_geo(outside, 'bus_stops_outside_seoul.gpkg')

    representative = representative[representative['within_seoul']].copy()
    representative['district_code'] = representative['district_code'].astype('string').str.replace(r'\.0$','',regex=True).str.zfill(5)

    save_geo(representative, 'bus_stops_year.gpkg')
    save_table(representative.drop(columns='geometry'), 'bus_stops_year.csv')
    save_table(pd.DataFrame(snapshot_audit), 'bus_snapshot_selection_audit.csv')

    print('✓ 7-1 버스정류장 전처리 완료')
    mark_done('bus')

else:
    print('기존 결과 재사용: bus')


# ============================================================
# 7-1A. 이전 결과파일 여부 안전검사
# ============================================================

bus = gpd.read_file(OUT/'bus_stops_year.gpkg').to_crs(CRS)

required_bus_cols = ['year','stop_id','node_id','ars_id','stop_name',
                     'district_code','district_name','geometry']
missing_bus_cols = [c for c in required_bus_cols if c not in bus.columns]

if missing_bus_cols:
    raise RuntimeError(
        f'기존 bus_stops_year.gpkg가 이전 코드로 생성되었습니다.\n'
        f'없는 컬럼: {missing_bus_cols}\n'
        f'FORCE_RUN에 "bus"를 추가한 뒤 7장을 처음부터 다시 실행하세요.'
    )

print('\n버스정류장 최종 기본 컬럼 확인:')
print(bus.columns.tolist())


# ============================================================
# 7-2. 스마트쉼터 원본 전처리
# ============================================================

smart_path = input_file('smart_shelter')
preview = pd.read_excel(smart_path, header=None, nrows=20)

header_keywords = ['정류소','정류장','ARS','시설명','설치장소','주소','설치년도','설치연도']
header_row = None

for i,row in preview.iterrows():
    text = ' '.join(row.dropna().astype(str))
    if sum(k in text for k in header_keywords) >= 2:
        header_row = i
        break

if header_row is None: raise ValueError('스마트쉼터 헤더를 찾지 못했습니다.')

smart = pd.read_excel(smart_path, header=header_row)
smart.columns = [norm(c).strip() for c in smart.columns]

def pick(keys):
    return next((c for c in smart.columns if any(k in norm(c).lower() for k in keys)), None)

ars_col = pick(['ars','정류소번호','정류장번호'])
node_col = pick(['node','노드'])
name_col = pick(['시설명','정류소명','정류장명'])
place_col = pick(['설치장소'])
address_col = pick(['상세주소','주소'])
install_col = pick(['설치년도','설치연도','설치년','설치일','설치일자'])
lon_col = pick(['경도','longitude','lon','x좌표'])
lat_col = pick(['위도','latitude','lat','y좌표'])

print('\n스마트쉼터 header_row:', header_row)
print('스마트쉼터 원본 컬럼:', smart.columns.tolist())
print('ARS:',ars_col,'NODE:',node_col,'NAME:',name_col,
      'PLACE:',place_col,'ADDRESS:',address_col,
      'INSTALL:',install_col,'LON:',lon_col,'LAT:',lat_col)

smart['smart_id'] = [f'SMART_{i:05d}' for i in range(len(smart))]
smart['ars_id'] = smart[ars_col].astype('string').str.replace(r'\.0$','',regex=True).str.strip() if ars_col else pd.NA
smart['node_id'] = smart[node_col].astype('string').str.replace(r'\.0$','',regex=True).str.strip() if node_col else pd.NA
smart['smart_name'] = smart[name_col].astype('string').str.strip() if name_col else pd.NA
smart['install_place'] = smart[place_col].astype('string').str.strip() if place_col else pd.NA
smart['address'] = smart[address_col].astype('string').str.strip() if address_col else pd.NA

for c in ['ars_id','node_id']:
    smart.loc[smart[c].isin(['','nan','None','<NA>']), c] = pd.NA

if install_col:
    smart['install_year'] = pd.to_numeric(
        smart[install_col].astype('string').str.extract(r'(20\d{2})', expand=False),
        errors='coerce'
    ).astype('Int64')
else:
    smart['install_year'] = pd.Series(pd.NA,index=smart.index,dtype='Int64')

smart['smart_lon'] = pd.to_numeric(smart[lon_col],errors='coerce') if lon_col else np.nan
smart['smart_lat'] = pd.to_numeric(smart[lat_col],errors='coerce') if lat_col else np.nan

print('\n[스마트쉼터 설치연도]')
print(smart['install_year'].value_counts(dropna=False).sort_index())

save_table(
    smart[['smart_id','ars_id','node_id','smart_name','install_place',
           'address','install_year','smart_lon','smart_lat']],
    'smart_shelter_clean.csv'
)


# ============================================================
# 7-3. 스마트쉼터 ↔ 버스정류장 직접 ID 매칭
# ============================================================

for c in ['ars_id','node_id']:
    bus[c] = bus[c].astype('string').str.replace(r'\.0$','',regex=True).str.strip()

# 스마트쉼터 원본에는 현재 ARS/NODE가 없으므로
# 이 부분은 향후 다른 자료에서 ID가 확보되는 경우를 위해 유지
if smart['ars_id'].notna().any():

    ars_lookup = bus[
        ['year','stop_id','ars_id','node_id','stop_name',
         'district_code','district_name','geometry']
    ].dropna(subset=['ars_id']).drop_duplicates(['year','ars_id'])

    smart_match = smart.merge(ars_lookup,on='ars_id',how='left',suffixes=('','_bus'))
    smart_match['match_method'] = np.where(smart_match['geometry'].notna(),'ars_id','unmatched')
    smart_match['match_distance_m'] = np.where(smart_match['geometry'].notna(),0.0,np.nan)

else:
    # ID가 없으므로 각 스마트쉼터 × 분석연도 형태로 확장
    smart_match = smart.merge(pd.DataFrame({'year':YEARS}), how='cross')
    smart_match['stop_id'] = pd.NA
    smart_match['stop_name'] = pd.NA
    smart_match['district_code'] = pd.NA
    smart_match['district_name'] = pd.NA
    smart_match['geometry'] = None
    smart_match['match_method'] = 'unmatched'
    smart_match['match_distance_m'] = np.nan


# NODE 보조매칭
if smart['node_id'].notna().any():
    unmatched = smart_match[smart_match['geometry'].isna()].copy()
    matched = smart_match[smart_match['geometry'].notna()].copy()

    if len(unmatched):
        node_lookup = bus[
            ['year','stop_id','node_id','stop_name',
             'district_code','district_name','geometry']
        ].dropna(subset=['node_id']).drop_duplicates(['year','node_id'])

        node_result = unmatched.drop(
            columns=['stop_id','stop_name','district_code','district_name','geometry'],
            errors='ignore'
        ).merge(node_lookup,on=['year','node_id'],how='left')

        node_result.loc[node_result['geometry'].notna(),'match_method'] = 'node_id'
        node_result.loc[node_result['geometry'].notna(),'match_distance_m'] = 0.0
        smart_match = pd.concat([matched,node_result],ignore_index=True)


# ============================================================
# 7-4. 정류장명/설치장소 기반 후보 탐색
# ============================================================
# 자동 confirmed용이 아니라 주소 지오코딩 전 QA용 후보이다.
# 정확히 같은 정류장명이 하나만 존재할 때만 candidate로 기록한다.
# ============================================================

def normalize_place(x):
    if pd.isna(x): return pd.NA
    x = str(x).strip().lower()
    x = re.sub(r'\([^)]*\)', '', x)
    x = re.sub(r'[^0-9a-zA-Z가-힣]', '', x)
    return x if x else pd.NA

bus['stop_name_key'] = bus['stop_name'].map(normalize_place)
smart_match['smart_name_key'] = smart_match['smart_name'].map(normalize_place)
smart_match['install_place_key'] = smart_match['install_place'].map(normalize_place)

name_candidate_rows = []

for r in smart_match[smart_match['geometry'].isna()].itertuples():
    keys = [k for k in [r.smart_name_key,r.install_place_key] if pd.notna(k)]
    if not keys: continue

    b = bus[bus['year'].eq(r.year)].copy()
    cand = b[b['stop_name_key'].isin(keys)].copy()

    if len(cand)==1:
        c = cand.iloc[0]
        name_candidate_rows.append({
            'smart_id':r.smart_id,
            'year':r.year,
            'smart_name':r.smart_name,
            'install_place':r.install_place,
            'address':r.address,
            'candidate_stop_id':c['stop_id'],
            'candidate_stop_name':c['stop_name'],
            'candidate_ars_id':c['ars_id'],
            'candidate_node_id':c['node_id'],
            'candidate_district':c['district_name'],
            'candidate_type':'unique_exact_name'
        })

name_candidates = pd.DataFrame(name_candidate_rows)
save_table(name_candidates,'smart_shelter_name_match_candidates.csv')

print('\n정류장명 단일 일치 후보:',len(name_candidates))
print('※ 이름 일치는 자동 confirmed 처리하지 않고 검토 후보로만 저장합니다.')


# ============================================================
# 7-5. 주소 지오코딩 필요 자료 분리
# ============================================================

unmatched_smart_ids = set(
    smart_match.loc[smart_match['geometry'].isna(),'smart_id'].dropna()
)

needs_geocoding = smart[
    smart['smart_id'].isin(unmatched_smart_ids)
].copy()

needs_geocoding['geocoding_query'] = needs_geocoding['address']

save_table(
    needs_geocoding[
        ['smart_id','smart_name','install_place','address',
         'geocoding_query','install_year']
    ],
    'smart_shelter_needs_geocoding.csv'
)

print('\n주소 지오코딩 필요 스마트쉼터:',len(needs_geocoding))
print('→ smart_shelter_needs_geocoding.csv 저장 완료')


# ============================================================
# 7-6. 좌표가 이미 존재하는 경우 공간 후보 탐색
# ============================================================
# 현재 원본은 LON/LAT가 없으므로 실행되지 않는 것이 정상.
# 향후 지오코딩 좌표가 추가되면 그대로 사용 가능.
# ============================================================

coord = smart[
    smart['smart_lon'].between(125,129) &
    smart['smart_lat'].between(33,39)
].copy()

if len(coord):

    smart_points = gpd.GeoDataFrame(
        coord,
        geometry=gpd.points_from_xy(coord.smart_lon,coord.smart_lat),
        crs='EPSG:4326'
    ).to_crs(CRS)

    spatial_parts = []

    for year in YEARS:
        b = bus[bus['year'].eq(year)].copy()
        if b.empty: continue

        near = gpd.sjoin_nearest(
            smart_points,
            b[['stop_id','ars_id','node_id','stop_name',
               'district_code','district_name','geometry']],
            how='left',
            distance_col='candidate_distance_m'
        )
        near['year'] = year
        spatial_parts.append(near)

    if spatial_parts:
        spatial_candidates = pd.concat(spatial_parts,ignore_index=True)
        save_table(
            spatial_candidates.drop(columns='geometry',errors='ignore'),
            'smart_shelter_spatial_match_candidates.csv'
        )

        print('좌표 기반 최근접 후보:',len(spatial_candidates))
        print('※ 거리 임계값 확정 전이므로 자동 confirmed 하지 않습니다.')


# ============================================================
# 7-7. 현재 확보된 직접매칭 결과의 역사적 상태
# ============================================================

smart_match['historical_status'] = np.select(
    [
        smart_match['geometry'].isna(),
        smart_match['install_year'].isna(),
        smart_match['year'] < smart_match['install_year']
    ],
    [
        'unknown',
        'historical_unknown',
        'not_yet_installed'
    ],
    default='confirmed'
)

smart_match['facility_type'] = 'smart_shelter'
smart_match['facility_verified'] = smart_match['historical_status'].eq('confirmed')

print('\n=== 현재 스마트쉼터 상태 ===')
display(
    smart_match['historical_status']
    .value_counts(dropna=False)
    .rename_axis('historical_status')
    .reset_index(name='count')
)

save_table(
    smart_match.drop(columns='geometry',errors='ignore'),
    'smart_shelter_match_audit.csv'
)


# ============================================================
# 7-8. 버스정류장 최종 데이터 생성
# ============================================================

final_bus = bus.drop(columns='stop_name_key',errors='ignore').copy()

matched_facility = smart_match[
    smart_match['stop_id'].notna()
].copy()

priority = {'confirmed':1,'not_yet_installed':2,'historical_unknown':3,'unknown':4}

if len(matched_facility):
    matched_facility['_priority'] = matched_facility['historical_status'].map(priority).fillna(99)

    matched_facility = (
        matched_facility
        .sort_values(['year','stop_id','_priority'])
        .drop_duplicates(['year','stop_id'],keep='first')
    )

    matched_facility = matched_facility[
        ['year','stop_id','facility_type','facility_verified',
         'install_year','historical_status','match_method','match_distance_m']
    ]

    final_bus = final_bus.merge(matched_facility,on=['year','stop_id'],how='left')

else:
    final_bus['facility_type'] = pd.NA
    final_bus['facility_verified'] = False
    final_bus['install_year'] = pd.Series(pd.NA,index=final_bus.index,dtype='Int64')
    final_bus['historical_status'] = 'unknown'
    final_bus['match_method'] = pd.NA
    final_bus['match_distance_m'] = np.nan

final_bus['facility_verified'] = final_bus['facility_verified'].fillna(False).astype(bool)
final_bus['historical_status'] = final_bus['historical_status'].fillna('unknown')
final_bus['install_year'] = pd.to_numeric(final_bus['install_year'],errors='coerce').astype('Int64')

duplicate_final = final_bus.duplicated(['year','stop_id']).sum()
print('\n최종 (year, stop_id) 중복:',duplicate_final)

if duplicate_final:
    raise ValueError('최종 버스정류장에 (year, stop_id) 중복이 있습니다.')

save_geo(final_bus,'bus_stops_final.gpkg')

final_csv = final_bus.drop(columns='geometry').copy()
final_csv['x_5186'] = final_bus.geometry.x
final_csv['y_5186'] = final_bus.geometry.y
save_table(final_csv,'bus_stops_final.csv')


# ============================================================
# 7-9. 전처리 QA
# ============================================================

audit = []

for year in YEARS:
    y = final_bus[final_bus['year'].eq(year)]

    audit.append({
        'year':year,
        'stop_count':len(y),
        'node_id_known':int(y['node_id'].notna().sum()),
        'ars_id_known':int(y['ars_id'].notna().sum()),
        'district_known':int(y['district_code'].notna().sum()),
        'confirmed':int(y['historical_status'].eq('confirmed').sum()),
        'not_yet_installed':int(y['historical_status'].eq('not_yet_installed').sum()),
        'historical_unknown':int(y['historical_status'].eq('historical_unknown').sum()),
        'unknown':int(y['historical_status'].eq('unknown').sum())
    })

bus_audit = pd.DataFrame(audit)
save_table(bus_audit,'bus_stop_preprocessing_audit.csv')

print('\n=== 버스정류장 전처리 QA ===')
display(bus_audit)

print('\n=== 연도별 시설상태 ===')
display(final_bus.groupby('year')['historical_status'].value_counts().unstack(fill_value=0))

print('\n[해석]')
print('- confirmed: 해당 연도 스마트쉼터 시설 존재 확인')
print('- not_yet_installed: 해당 연도에는 설치 전')
print('- historical_unknown: 정류장 연결은 됐지만 설치연도 불명')
print('- unknown: 현재 자료만으로 시설 존재 여부 확인 불가')
print('- unknown은 그늘 없음이 아님')
print('- 이름 일치 후보는 자동 확정하지 않음')
print('- 주소 지오코딩 후 공간매칭이 필요한 자료는 별도 CSV로 저장')

print('\n✓ 7장 버스정류장 전처리 완료')


[2021] 원본 컬럼: ['STDR_DE', 'NODE_ID', 'STTN_NO', 'STTN_NM', 'CRDNT_X', 'CRDNT_Y', 'STTN_TY', 'Unnamed: 7']
[2021] 원본 행: 149640
[2021] 좌표 유효: 149640
[2021] 좌표 비정상: 0
[2021] 중복 제거: 0

[2022] 원본 컬럼: ['STDR_DE', 'NODE_ID', 'STTN_NO', 'STTN_NM', 'CRDNT_X', 'CRDNT_Y', 'STTN_TY', 'Unnamed: 7']
[2022] 원본 행: 150218
[2022] 좌표 유효: 150218
[2022] 좌표 비정상: 0
[2022] 중복 제거: 0

[2023] 원본 컬럼: ['STDR_DE', 'NODE_ID', 'STTN_NO', 'STTN_NM', 'CRDNT_X', 'CRDNT_Y', 'STTN_TY', 'Unnamed: 7']
[2023] 원본 행: 150955
[2023] 좌표 유효: 150955
[2023] 좌표 비정상: 0
[2023] 중복 제거: 0

[2024] 원본 컬럼: ['NODE_ID', 'ARS_ID', '정류소명', 'X좌표', 'Y좌표', '정류소타입']
[2024] 원본 행: 11262
[2024] 좌표 유효: 11262
[2024] 좌표 비정상: 0
[2024] 중복 제거: 0

[2025] 원본 컬럼: ['NODE_ID', 'ARS_ID', '정류소명', 'X좌표', 'Y좌표', '정류소타입']
[2025] 원본 행: 11245
[2025] 좌표 유효: 11245
[2025] 좌표 비정상: 0
[2025] 중복 제거: 0
저장: bus_stops_clean.gpkg 도형: 473320

[서울 자치구 공간조인]
서울 내부: 59994
서울 경계 연결 실패: 94
저장: bus_stops_outside_seoul.gpkg 도형: 94
저장: bus_stops_year.gpkg 도형: 59994
저장: bus_stops_year.csv 

,historical_status,count
0,unknown,995


저장: smart_shelter_match_audit.csv 행: 995

최종 (year, stop_id) 중복: 0
저장: bus_stops_final.gpkg 도형: 59994
저장: bus_stops_final.csv 행: 59994
저장: bus_stop_preprocessing_audit.csv 행: 5

=== 버스정류장 전처리 QA ===


,year,stop_count,node_id_known,ars_id_known,district_known,confirmed,not_yet_installed,historical_unknown,unknown
0,2021,12451,12451,12451,12451,0,0,0,12451
1,2022,12501,12501,12501,12501,0,0,0,12501
2,2023,12563,12563,12563,12563,0,0,0,12563
3,2024,11247,11247,11247,11247,0,0,0,11247
4,2025,11232,11232,11232,11232,0,0,0,11232



=== 연도별 시설상태 ===


historical_status,unknown
year,
2021,12451
2022,12501
2023,12563
2024,11247
2025,11232



[해석]
- confirmed: 해당 연도 스마트쉼터 시설 존재 확인
- not_yet_installed: 해당 연도에는 설치 전
- historical_unknown: 정류장 연결은 됐지만 설치연도 불명
- unknown: 현재 자료만으로 시설 존재 여부 확인 불가
- unknown은 그늘 없음이 아님
- 이름 일치 후보는 자동 확정하지 않음
- 주소 지오코딩 후 공간매칭이 필요한 자료는 별도 CSV로 저장

✓ 7장 버스정류장 전처리 완료


## 8장. 스마트쉼터 · 인공 그늘막
스마트쉼터는 7장에서 연도별 상태를 반영합니다. **2026 인공 그늘막은 2025 공급으로 소급하지 않으며**, 2025 후보지에 2026 시설을 추가한 뒤 남는 공백을 재검토하는 용도로만 사용합니다.


In [53]:
# facilities: 결과가 있으면 기본적으로 재사용합니다. 다시 처리하려면 FORCE_RUN에 이름을 추가하세요.
if should_run('facilities', []):
    for key in ['스마트쉼터 현황','폭염저감시설 목록']:
        p=input_file('smart_shelter' if '스마트쉼터' in key else 'sunshade')
        print('\n',norm(p.name))
        display(pd.read_excel(p,header=None,nrows=12))
    # 시설 데이터의 실제 헤더 행과 좌표 컬럼을 확인한 후 아래 설정값을 지정합니다.
    FACILITY_CONFIG={
        'smart_shelter': {'keyword':EXPECTED_INPUTS['smart_shelter'],'header_row':None,'lon_col':None,'lat_col':None,'date_label':'2025-05-20'},
        'sunshade_2026': {'keyword':EXPECTED_INPUTS['sunshade'],'header_row':None,'lon_col':None,'lat_col':None,'date_label':'2026-09-10'}
    }
    facility_parts=[]
    for kind,cfg in FACILITY_CONFIG.items():
        if cfg['header_row'] is None:
            print(kind,'헤더 확인 대기 — 위 원본 미리보기를 보고 설정하세요.');continue
        d=pd.read_excel(find_file(cfg['keyword'],'.xlsx'),header=cfg['header_row'])
        d['facility_type']=kind;d['snapshot_date']=cfg['date_label']
        if cfg['lon_col'] and cfg['lat_col']:
            d['lon']=pd.to_numeric(d[cfg['lon_col']],errors='coerce')
            d['lat']=pd.to_numeric(d[cfg['lat_col']],errors='coerce')
            d=d[d.lon.between(125,129)&d.lat.between(33,39)]
            facility_parts.append(gpd.GeoDataFrame(d,geometry=gpd.points_from_xy(d.lon,d.lat),crs='EPSG:4326').to_crs(CRS))
        else:
            save_table(d,f'{kind}_needs_geocoding.csv')
    if facility_parts: save_geo(pd.concat(facility_parts,ignore_index=True),'shade_facilities.gpkg')
    if facility_parts:
        mark_done('facilities')
    else:
        print('시설 좌표·헤더 확인 전이므로 완료 처리하지 않습니다.')
else:
    print('기존 결과 재사용: facilities')


 스마트쉼터 현황(20250520).xlsx


,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,스마트쉼터 현황,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,시도,시군구,시설명,설치장소,상세주소,설치년도
4,서울,-,스마트셸터,14-015 홍대입구역(중) 버스정류소,서울 마포구 동교동 155-55,2021
5,서울,-,스마트셸터,14-016 홍대입구역(중) 버스정류소,서울 마포구 동교동 155-55,2021
6,서울,-,스마트셸터,14-011 합정역(중) 버스정류소,서울 마포구 합정동 402,2021
7,서울,-,스마트셸터,14-012 합정역(중) 버스정류소,서울 마포구 합정동 402,2021
8,서울,-,스마트셸터,02-008 숭례문(중) 버스정류소,서울 중구 남대문로5가 84-15,2021
9,서울,-,스마트셸터,05-232 건대입구역사거리.건대병원 버스정류소,서울 광진구 화양동 4-15,2021



 폭염저감시설 목록_그늘막(260910).xlsx


,0,1,2,3,4,5,6,7,8,9,10,11
0,폭염저감시설 관리 현황,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,기준년도 : 2026년,NaN,NaN,시설종류 : 그늘막,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,관할지자체 : 서울특별시,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,발행일자 : 2026-09-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,연번,종류,시도,시군구,NaN,관리번호,설치장소명,설치위치,NaN,경도,위도,설치일시
5,NaN,NaN,NaN,NaN,읍면동,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,(행정동),NaN,NaN,도로명주소,지번주소,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,그늘막(고정형),서울특별시,종로구,청운효자동,9(청운효자동-1),청운효자동주민센터,서울특별시 종로구 자하문로 92 (궁정동),서울특별시 종로구 궁정동 12-1,126.9706098,37.5840822,2026-07-14
9,2,그늘막(고정형),서울특별시,종로구,사직동,1(사직동-1),사직동주민센터 앞 교통섬,서울특별시 종로구 사직로9길 5 (사직동),서울특별시 종로구 사직동 1-28,126.9689218,37.5761763,2026-07-14


smart_shelter 헤더 확인 대기 — 위 원본 미리보기를 보고 설정하세요.
sunshade_2026 헤더 확인 대기 — 위 원본 미리보기를 보고 설정하세요.
시설 좌표·헤더 확인 전이므로 완료 처리하지 않습니다.


## 9장. 통합 분석 전 필수 점검
현재 단계에서는 최종 충분성·형평성·후보지를 만들지 않습니다. 먼저 공간 LST와 연도별 시설 존재정보가 필요합니다.

권장 최종 구조:
1. `trend_gu_year`: 2021~2025 자치구 추세
2. 2025 세부 공간 LST + 실제 공급공간으로 후보지 탐색
3. 충분성 부족 ∩ 형평성 부족
4. 2026 인공 그늘막을 겹쳐 잔여 공백 재검토


In [54]:
print('현재 구현 완료 범위: 자치구 경계, 기준시간, 고령 생활인구, 가로수, 건물 높이/선택 그림자, 버스·스마트쉼터 전처리')
print('필수 미구현: 공간 LST, 연도별 인공그늘막 공급, 수요/공급 통합, 충분성·형평성, 2025 후보지, 2026 재검토')
print('주의: 고령 생활인구를 500m 격자에 임의 배분하지 않습니다.')


현재 구현 완료 범위: 자치구 경계, 기준시간, 고령 생활인구, 가로수, 건물 높이/선택 그림자, 버스·스마트쉼터 전처리
필수 미구현: 공간 LST, 연도별 인공그늘막 공급, 수요/공급 통합, 충분성·형평성, 2025 후보지, 2026 재검토
주의: 고령 생활인구를 500m 격자에 임의 배분하지 않습니다.


## 10장. 최종 품질검증 및 연구 한계
- **공간단위:** 2021~2025 형평성·추세의 기본 단위는 자치구. 세부 격자는 2025 후보지 탐색용 보조단위입니다.
- **가로수:** 2025 실제 좌표와 연도별 자치구 총량을 분리합니다. 반경 버퍼/공급지수는 실제 그늘면적이 아닙니다.
- **건물:** 1월 OSM 배치가 여름까지 유지되었다는 가정. 2026 높이 역적용은 동일성 검증 필수. 높이 미확인은 `unknown`입니다.
- **버스정류소:** 정류장 위치와 차양 존재는 별개입니다. 스마트쉼터는 설치연도 이전으로 소급하지 않습니다.
- **생활인구:** 2021~2023은 65+, 2024~2025는 65~74이므로 2023→2024 직접 증감 해석을 금지합니다.
- **ASOS:** 기준시간 선정용이며 공간 LST의 대체물이 아닙니다.
- **2026 그늘막:** 2025 공급으로 소급하지 않고 최종 후보지 재검토에만 사용합니다.
- 모든 추정 결과는 기준일·가정·자료 출처·미확인 비율을 함께 보고합니다.


In [55]:
print('결과 저장 폴더:',OUT)
print('생성된 파일:')
for p in sorted(OUT.iterdir()):
    if p.is_file(): print(' -',p.name)
print('미완료 항목은 데이터 검증/연구팀 의사결정이 필요한 단계입니다.')

결과 저장 폴더: /Users/hongbini/Documents/vscode_HB/교학/data/processed
생성된 파일:
 - asos_hourly.csv
 - asos_summer_clean.parquet
 - building_extraction_audit.csv
 - building_floor_height_medians.csv
 - building_gis_best_matches.parquet
 - building_gis_candidates_2021.parquet
 - building_gis_candidates_2022.parquet
 - building_gis_candidates_2023.parquet
 - building_gis_candidates_2024.parquet
 - building_gis_candidates_2025.parquet
 - building_height_audit.csv
 - building_identity_review.csv
 - building_identity_review_template.csv
 - building_registry_verified.csv
 - buildings_final_2021.gpkg
 - buildings_final_2022.gpkg
 - buildings_final_2023.gpkg
 - buildings_final_2024.gpkg
 - buildings_final_2025.gpkg
 - buildings_gis_2026_clean.gpkg
 - buildings_osm_2021.gpkg
 - buildings_osm_2022.gpkg
 - buildings_osm_2023.gpkg
 - buildings_osm_2024.gpkg
 - buildings_osm_2025.gpkg
 - bus_snapshot_selection_audit.csv
 - bus_stop_preprocessing_audit.csv
 - bus_stops_clean.gpkg
 - bus_stops_final.csv
 - bu

## Mac 실행 및 재실행 안내
- 일부 단계만 재실행할 때는 0장의 `FORCE_RUN`에 단계명을 넣습니다.
- 가로수 반경을 바꾸면 `tree_allocation`을 재실행합니다.
- 건물 매칭 임계값은 연구 설정값입니다. 수동검증 후 확정하고 관련 `building_*` 단계를 재실행합니다.
- `CREATE_CANDIDATE_GRID=True`는 공간 LST가 준비된 뒤 후보지 분석을 시작할 때만 사용합니다.
- 2026 인공 그늘막은 2025 시설로 역적용하지 않습니다.


## 마지막장. 전처리·분석 상태 점검
반드시 맨 마지막에 실행합니다. 파일 존재는 내용 검증 완료를 의미하지 않습니다.

권장 실행 순서: 0장 → 자치구 경계 → ASOS → 고령 생활인구 → 가로수 → 건물 5-0~5-8 → 동일성/대장 검토 → 선택 그림자 → 버스·스마트쉼터.
그 다음 **공간 LST를 추가한 후** 수요·공급 통합 분석을 별도 구현합니다.


In [56]:
from pathlib import Path
import numpy as np,pandas as pd
from IPython.display import display
if 'OUT' not in globals():raise RuntimeError('먼저 0장 경로설정을 실행하세요.')
CORE_OUTPUTS=[
'districts.gpkg','asos_summer_clean.parquet','asos_hourly.csv','reference_time.csv',
'living_elderly_gu_year.parquet','trees_clean.gpkg','tree_totals_verified.csv','tree_district_year.parquet',
'tree_supply_sensitivity.csv','tree_assumed_influence_2025.gpkg',
'buildings_2021.gpkg','buildings_2022.gpkg','buildings_2023.gpkg','buildings_2024.gpkg','buildings_2025.gpkg',
'building_extraction_audit.csv','building_height_audit.csv','bus_stops_year.gpkg','bus_stop_shade_status.parquet',
'smart_shelters_matched.gpkg','smart_shelter_match_audit.csv']
OPTIONAL_AFTER_HEIGHT=['building_shadow_year.gpkg','building_shade_district_year.parquet','building_solar_positions.csv']
FUTURE_REQUIRED=['spatial_lst_dataset','annual_sunshade_supply','demand_supply_integration','adequacy_equity_results','candidate_2025','candidate_after_2026_sunshade']
rows=[]
for name in CORE_OUTPUTS+OPTIONAL_AFTER_HEIGHT:
    p=Path(OUT)/name;exists=p.is_file();size=p.stat().st_size if exists else 0
    rows.append({'file':name,'stage':'선택: 높이검증 후' if name in OPTIONAL_AFTER_HEIGHT else '현재 전처리','exists':exists,'nonempty':size>0,'size_mb':round(size/1024**2,2) if exists else np.nan})
audit=pd.DataFrame(rows);display(audit);audit.to_csv(Path(OUT)/'pipeline_output_status.csv',index=False,encoding='utf-8-sig')
print('\n[연구상 아직 필요한 단계]')
for x in FUTURE_REQUIRED:print(' -',x)
# 핵심 자치구×연도 산출물 검증
for name in ['living_elderly_gu_year.parquet','tree_district_year.parquet','building_shade_district_year.parquet']:
    p=Path(OUT)/name
    if not p.exists():continue
    d=pd.read_parquet(p)
    if {'year','district_code'}.issubset(d.columns):
        dup=d.duplicated(['year','district_code']).sum();n_gu=d.groupby('year')['district_code'].nunique()
        print(f'\n{name}: 행={len(d)}, 키중복={dup}');print(n_gu)
        if name!='building_shade_district_year.parquet' and (dup or not n_gu.eq(25).all()):print('⚠ 연도×25구 구조를 확인하세요.')
print('\n※ 고령 생활인구를 세부 격자에 임의 배분하지 않습니다.')
print('※ 가로수 공급지수는 실제 그늘면적이 아닙니다.')
print('※ unknown을 no-shade로 바꾸지 않습니다.')


,file,stage,exists,nonempty,size_mb
0,districts.gpkg,현재 전처리,True,True,0.91
1,asos_summer_clean.parquet,현재 전처리,True,True,0.11
2,asos_hourly.csv,현재 전처리,True,True,0.01
3,reference_time.csv,현재 전처리,True,True,0.00
4,living_elderly_gu_year.parquet,현재 전처리,False,False,NaN
5,trees_clean.gpkg,현재 전처리,True,True,45.17
6,tree_totals_verified.csv,현재 전처리,True,True,0.00
7,tree_district_year.parquet,현재 전처리,True,True,0.01
8,tree_supply_sensitivity.csv,현재 전처리,True,True,0.02
9,tree_assumed_influence_2025.gpkg,현재 전처리,True,True,365.55



[연구상 아직 필요한 단계]
 - spatial_lst_dataset
 - annual_sunshade_supply
 - demand_supply_integration
 - adequacy_equity_results
 - candidate_2025
 - candidate_after_2026_sunshade

tree_district_year.parquet: 행=125, 키중복=0
year
2021    25
2022    25
2023    25
2024    25
2025    25
Name: district_code, dtype: int64

※ 고령 생활인구를 세부 격자에 임의 배분하지 않습니다.
※ 가로수 공급지수는 실제 그늘면적이 아닙니다.
※ unknown을 no-shade로 바꾸지 않습니다.
